# Archived analysis workflow

This notebook records the author workflow used during revision. It requires the corresponding external model-ready inputs, saved predictions and/or checkpoints. Its `/content/drive/MyDrive` paths describe the author environment and must be reviewed before execution. Do not run all cells merely to inspect the repository. See [analysis/README.md](../README.md) and the main repository README for the supported benchmark entry point.


# SpaGAT R1.5：距离匹配与固定注意力的消息屏蔽（v3）

**Colab → 更改运行时类型 → GPU（T4 即可）→ 按顺序运行全部单元格。无需手动找文件或安装包。**

沿用已核实的 `liver_free_k4_3epochs/free_seed123_best.pth`，读取先前 R3.5/7 保存的预测来核验新推理。**不训练、不覆盖旧文件**；此次会新增固定模型推理。

### 本次修订原因

v2 返回了完整复现诊断：76,753 × 1,000 个 target 全部逐项相同；预测最大绝对差为 `7.7188e-6`，平均绝对差为 `4.3698e-8`。只有两行中的五个预测值超过 `atol=3e-6, rtol=2e-5`，从而触发停止；当时尚未启动正式控制分析。

根据这些实际差异，v3 将**旧预测复现**的绝对容差设为 `1e-5`，相对容差仍为 `2e-5`；并在控制开始前增加两类合格 receiver 的整体 MSE 对照，要求新旧 MSE 绝对差不超过 `1e-6`。旧严格容差下的超限数量仍会保留在报告中。target 仍须逐项相同，完整模型消息计算对照仍使用原先较严容差。此项修改只涉及数值复现检查，不根据任何 sender 效应结果调整标准。

保留旧导出的 CSV 读取方式、全部 76,753 个 tumor receiver 的原始顺序及 batch=32 分组。实际控制和指标仍仅针对原 26,551 个合格 receiver 及其中距离匹配成功者。模型、距离匹配规则、mask seed 和重复次数不变。核对通过后自动进行控制分析，失败则保存诊断并停止。

这两项控制检验：两类 tumor sender 的扰动差异是否仍存在于相似距离范围，以及差异是否依赖剩余边的 softmax 重新归一化。当前结果尚未生成，不预设 sender 排名应保持或改变。

### 对照设计（已预先固定）

- 每个 receiver 的 49 个邻居按距离分层：以最远邻居距离为半径，使用 20 个等宽区间，每个宽度为半径的 5%。原始坐标不转换，也不猜测微米单位。
- 每个区间同时存在 tumor_1/tumor_2 时，取两者数量的较小值。两类 sender 和随机对照使用相同的区间内数量；位置 0 的 receiver 永不屏蔽。
- **数量对照**采用相同的 receiver、相同的总屏蔽数量，但不匹配距离。因此新增距离控制与数量对照的差别不会夹杂屏蔽数量差异。
- 每组选中的 sender 同时用于两种操作：原有边排除/softmax 重新归一化，以及保留原始 softmax 权重、仅将选定 sender 的消息证据置零。共享偏置和模型权重保持原值。
- 两类 receiver、两种选取方式、三种 sender 条件、两种操作、10 次邻居抽样。三种 sender 条件为 tumor_1、tumor_2、random；10 次不是训练 seeds。

只分析两种 tumor sender 在距离上有共同支持的 receiver，报告排除数量和比例，不按结果调整距离区间。表达强度和移除比例会输出供核对，但不宣称它们已被匹配，也不把该分析称为因果验证。


## 1. 挂载 Drive

In [ ]:
from google.colab import drive
drive.mount("/content/drive")


## 2. 准备冻结的模型与分析代码

代码已内嵌。本地通过了匹配规则、消息运算公式、汇总、批次索引和差异诊断检查，并使用 v2 返回的真实失败批次核对了新标准。首轮 Colab 的消息计算对照已通过；v3 的完整真实数据运行仍需在下一步执行。


In [ ]:
from pathlib import Path
import sys, hashlib, uuid, json
MODULES = {'spagat_r15/__init__.py': '"""\nSpaGAT package.\n"""\n\nfrom .gene_program_model import SpaGP, SpaGP_Loss\nfrom .dataloader import SPAGAT_dataset\n\n__all__ = [\n    "SpaGP",\n    "SpaGP_Loss",\n    "SPAGAT_dataset",\n]', 'spagat_r15/attention.py': 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom .embedding import FFN\n\ndef rho(x):\n    return torch.sqrt(F.relu(x)) - torch.sqrt(F.relu(-x))\n\ndef get_diagonal(x):\n    index = torch.arange(0, x.shape[1], 1)\n    return x[:, index, index, :]\n\n\ndef _normalize_pairwise_tensors(node, edge, embedding, edge_dim=None):\n    """Normalize Embedding\'s initial singleton receiver axis.\n\n    ``Embedding.forward`` supplies pairwise tensors as ``(B, 1, N, E)``: one\n    sender-wise edge representation shared by every receiver.  GRIT attention\n    operates on an explicit receiver/sender representation ``(B, N, N, E)``.\n    Expanding the singleton receiver axis before Q/K--edge arithmetic prevents\n    PyTorch from broadcasting the batch dimension into an unintended\n    ``(B, B, N, N, E)`` tensor.  Later GRIT layers already use the expanded\n    four-dimensional convention.\n    """\n    if node.ndim != 3:\n        raise ValueError(f"node must have shape (B, N, node_dim); got {tuple(node.shape)}.")\n    B, N, _ = node.shape\n\n    def normalize(name, tensor):\n        if tensor.ndim != 4:\n            raise ValueError(\n                f"{name} must have shape (B, 1, N, edge_dim) or (B, N, N, edge_dim); "\n                f"got {tuple(tensor.shape)}."\n            )\n        if tensor.shape[0] != B or tensor.shape[2] != N:\n            raise ValueError(\n                f"{name} batch and sender dimensions must be B={B} and N={N}; "\n                f"got {tuple(tensor.shape)}."\n            )\n        if tensor.shape[1] == 1:\n            tensor = tensor.expand(B, N, N, tensor.shape[-1])\n        elif tensor.shape[1] != N:\n            raise ValueError(\n                f"{name} receiver dimension must be 1 or N={N}; got {tuple(tensor.shape)}."\n            )\n        return tensor\n\n    edge = normalize("edge", edge)\n    embedding = normalize("embedding", embedding)\n    if edge.shape[-1] != embedding.shape[-1]:\n        raise ValueError(\n            "edge and embedding must have the same feature dimension; got "\n            f"{edge.shape[-1]} and {embedding.shape[-1]}."\n        )\n    if edge_dim is not None and edge.shape[-1] != edge_dim:\n        raise ValueError(\n            f"edge and embedding feature dimension must be edge_dim={edge_dim}; "\n            f"got {edge.shape[-1]}."\n        )\n    return edge, embedding\n\nclass GRIT_attention(nn.Module):\n    def __init__(self, node_dim, edge_dim, att_dim=8, program_aware=False, routing_mode="program"):\n        super().__init__()\n        self.program_aware = program_aware\n        if routing_mode not in {"grit", "program"}:\n            raise ValueError("routing_mode must be \'grit\' or \'program\'.")\n        self.routing_mode = routing_mode\n        self.node_dim = node_dim\n        self.base_edge_dim = edge_dim\n        self.edge_dim = edge_dim * 2\n        self.W_Q = nn.Linear(node_dim, self.edge_dim * att_dim)\n        self.W_K = nn.Linear(node_dim, self.edge_dim * att_dim)\n        self.W_V = nn.Linear(node_dim, node_dim)\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\n        self.W_En = nn.Linear(edge_dim, node_dim)\n        self.W_A = nn.Linear(edge_dim, 1, bias=False)\n        self.W_No = nn.Linear(node_dim, node_dim)\n        self.W_Eo = nn.Linear(edge_dim, edge_dim, bias=False)\n\n        # Program tokens live in the common edge-feature space (edge_dim).  They\n        # are supplied by a parent model so that token k has the same biological\n        # identity in every graph layer; these projections remain layer-specific.\n        if self.program_aware:\n            self.W_program_edge = nn.Linear(edge_dim, edge_dim, bias=False)\n            self.W_program_receiver = nn.Linear(node_dim, edge_dim, bias=False)\n            self.W_program_gate = nn.Linear(node_dim, edge_dim, bias=False)\n\n    def forward(self, x, program_tokens=None, program_edge_mask=None):\n        """Run edge-aware attention, optionally routed through communication programs.\n\n        Args:\n            x: ``[node, edge, embedding]`` with node ``(B, N, node_dim)`` and\n                broadcastable edge/embedding tensors ``(B, 1 or N, N, edge_dim)``.\n            program_tokens: Optional shared token bank ``(K, edge_dim)``.  A\n                token is a globally named latent communication program; the\n                same bank should be passed to every program-aware GRIT layer.\n            program_edge_mask: Optional inference-only ``(B, N, N, K)`` mask.\n                Zero entries remove sender edges from a program\'s softmax.\n\n        Returns:\n            ``[node_updated, edge_updated, program_state]``.  ``program_state``\n            is ``None`` in baseline mode, preserving the original computation.\n        """\n        node, edge, embedding = x\n        B, N, C = node.shape\n\n        # Always normalize the initial Embedding output, including the legacy\n        # path.  The routing mode must not determine whether Q/K arithmetic is\n        # performed on a correctly shaped pairwise edge tensor.\n        edge, embedding = _normalize_pairwise_tensors(\n            node, edge, embedding, edge_dim=self.base_edge_dim\n        )\n\n        if program_tokens is not None and not self.program_aware:\n            raise ValueError(\n                "program_tokens were provided, but this GRIT_attention was "\n                "constructed with program_aware=False."\n            )\n        if program_tokens is not None and program_tokens.ndim != 2:\n            raise ValueError(\n                "program_tokens must have shape (n_programs, edge_dim); got "\n                f"{tuple(program_tokens.shape)}."\n            )\n        if program_tokens is not None and program_tokens.shape[1] != self.base_edge_dim:\n            raise ValueError(\n                "program_tokens must have shape (n_programs, edge_dim), where "\n                f"edge_dim={self.base_edge_dim}; got {tuple(program_tokens.shape)}."\n            )\n        if program_edge_mask is not None and program_tokens is None:\n            raise ValueError("program_edge_mask requires program-aware routing tokens.")\n\n        Q = self.W_Q(node).reshape(B, N, -1, self.edge_dim).permute(0, 3, 1, 2)\n        K = self.W_K(node).reshape(B, N, -1, self.edge_dim).permute(0, 3, 2, 1)\n        QK = (Q @ K).permute(0, 2, 3, 1)\n        edge = F.gelu(rho((QK[:, :, :, :self.edge_dim // 2]) * self.W_Ew(edge)) + self.W_Eb(edge) + QK[:, :, :, self.edge_dim // 2:] + embedding)\n        V = self.W_V(node)\n\n        if program_tokens is None:\n            # This is the original GRIT message-passing path, kept numerically\n            # unchanged when no shared program bank is supplied.\n            alpha = self.W_A(edge).squeeze(dim=-1)\n            alpha = F.softmax(alpha, dim=-1)\n            node = alpha @ V + self.W_En(torch.sum(edge * alpha.unsqueeze(dim=-1), dim=-2))\n            node = self.W_No(node)\n            edge = self.W_Eo(edge)\n            return [node, edge, None]\n\n        if self.routing_mode == "grit":\n            if program_tokens.shape[0] != 1:\n                raise ValueError("routing_mode=\'grit\' is defined for the controlled K=1 ablation only.")\n            # Controlled routing ablation: retain GRIT\'s original scalar edge\n            # routing and expose it as the K=1 program-state interface.  The\n            # downstream program gate/messages/free decoder are unchanged.\n            program_attention = F.softmax(self.W_A(edge).squeeze(dim=-1), dim=-1).unsqueeze(dim=-1)\n        else:\n            # Program routing: the edge, receiver, and shared program token\n            # interact multiplicatively. This is the pre-existing SpaGP score.\n            edge_program = self.W_program_edge(edge)\n            receiver_program = self.W_program_receiver(node)\n            program_scores = torch.einsum(\n                "bije,bie,ke->bijk", edge_program, receiver_program, program_tokens\n            ) / (self.base_edge_dim ** 0.5)\n            program_attention = None\n        if program_edge_mask is not None:\n            expected_shape = (B, N, N, program_tokens.shape[0])\n            if tuple(program_edge_mask.shape) != expected_shape:\n                raise ValueError(\n                    f"program_edge_mask must have shape {expected_shape}; got "\n                    f"{tuple(program_edge_mask.shape)}."\n                )\n            allowed = program_edge_mask.to(device=program_scores.device, dtype=torch.bool)\n            if not torch.all(allowed.any(dim=2)):\n                raise ValueError("program_edge_mask removes every sender for at least one receiver/program.")\n            # This is inference-time graph intervention: masked edges receive\n            # exactly zero attention and remaining edges are renormalized by\n            # the existing neighbor softmax.\n            if self.routing_mode == "grit":\n                grit_scores = self.W_A(edge).squeeze(dim=-1).masked_fill(~allowed[..., 0], float("-inf"))\n                program_attention = F.softmax(grit_scores, dim=-1).unsqueeze(dim=-1)\n            else:\n                program_scores = program_scores.masked_fill(~allowed, float("-inf"))\n                program_attention = F.softmax(program_scores, dim=2)\n        elif program_attention is None:\n            program_attention = F.softmax(program_scores, dim=2)\n\n        # Aggregate node and edge evidence independently for each program:\n        # m[i, k] = sum_j alpha[i, j, k] * message[i, j].\n        node_messages = torch.einsum("bijk,bjd->bikd", program_attention, V)\n        edge_messages = torch.einsum("bijk,bije->bike", program_attention, edge)\n        program_messages = node_messages + self.W_En(edge_messages)\n\n        # Receiver-specific, non-negative gates fuse the K program messages back\n        # into the original node_dim, keeping the downstream tensor shape intact.\n        gate_scores = torch.einsum(\n            "bin,kn->bik", self.W_program_gate(node), program_tokens\n        ) / (self.base_edge_dim ** 0.5)\n        program_activations = F.softplus(gate_scores)\n        node = torch.sum(program_messages * program_activations.unsqueeze(dim=-1), dim=2)\n        node = self.W_No(node)\n        edge = self.W_Eo(edge)\n\n        program_state = {\n            "attention": program_attention,\n            "messages": program_messages,\n            "activations": program_activations,\n        }\n        return [node, edge, program_state]\n\nclass DegScaler(nn.Module):\n    def __init__(self, node_dim):\n        super().__init__()\n        scaler = (2 / node_dim) ** 0.5\n        self.theta1 = nn.Parameter(torch.randn(node_dim) * scaler)\n        self.theta2 = nn.Parameter(torch.randn(node_dim) * scaler)\n\n    def forward(self, node, degree):\n        node = self.theta1 * node + torch.log(degree + 1).unsqueeze(dim=-1) * node * self.theta2\n        return node\n\nclass Multi_Head_Attention(nn.Module):\n    def __init__(self, node_dim, edge_dim, num_heads, att_dim=8):\n        super().__init__()\n        self.attentions = nn.ModuleList([\n            GRIT_attention(node_dim, edge_dim, att_dim) for _ in range(num_heads)\n        ])\n        self.W_hn = nn.Parameter(torch.ones(num_heads, 1) / num_heads)\n        self.W_he = nn.Parameter(torch.ones(num_heads, 1) / num_heads)\n\n    def forward(self, x):\n        results = [attentioni(x) for attentioni in self.attentions]\n        node = (torch.stack([tmp[0] for tmp in results], dim=-1) @ self.W_hn).squeeze(dim=-1)\n        edge = torch.stack([tmp[1] for tmp in results], dim=-1) @ self.W_he\n        edge = edge.squeeze(dim=-1)\n        return [node, edge, x[2]]\n\nclass GRIT_encoder(nn.Module):\n    def __init__(self, node_dim, edge_dim, num_heads, att_dim=8, program_aware=False, routing_mode="program"):\n        super().__init__()\n        self.program_aware = program_aware\n        self.edge_dim = edge_dim\n        self.routing_mode = routing_mode\n        self.attentions = Multi_Head_Attention(node_dim, edge_dim, num_heads, att_dim)\n        if program_aware:\n            # Keep Multi_Head_Attention unchanged for baseline callers, but replace\n            # its heads with token-aware GRIT attention modules for this encoder.\n            # The shared tokens themselves are supplied to forward(), not owned by\n            # a layer, so program index k remains consistent across GNN layers.\n            self.attentions.attentions = nn.ModuleList([\n                GRIT_attention(node_dim, edge_dim, att_dim, program_aware=True, routing_mode=routing_mode)\n                for _ in range(num_heads)\n            ])\n        self.FFN = FFN(node_dim)\n        self.ln1 = nn.LayerNorm(node_dim)\n        self.ln2 = nn.LayerNorm(node_dim)\n        self.ln_edge = nn.LayerNorm(edge_dim)\n\n    def forward(self, x, program_tokens=None, program_edge_mask=None):\n        """Encode one graph layer, optionally with shared program-aware routing.\n\n        Without ``program_tokens``, this follows the original GRIT encoder path\n        and returns ``[node, edge, distance_embedding]``.  With a shared token\n        bank, each head returns program-specific routing state; the encoder\n        combines it across heads and returns it as a fourth element.\n        """\n        node, edge, distance = x\n        # Keep the residual edge tensor in the same four-dimensional convention\n        # as GRIT_attention.  This is essential for the baseline residual add.\n        edge, distance = _normalize_pairwise_tensors(\n            node, edge, distance, edge_dim=self.edge_dim\n        )\n        x = [node, edge, distance]\n        if program_tokens is None:\n            # Preserve the existing multi-head aggregation and output contract.\n            x = self.attentions(x)\n            edge = self.ln_edge(edge + x[1])\n            node = self.ln1(node + x[0])\n            node = self.ln2(node + self.FFN(node))\n            return [node, edge, distance]\n\n        if not self.program_aware:\n            raise ValueError(\n                "program_tokens were provided, but this GRIT_encoder was "\n                "constructed with program_aware=False."\n            )\n\n        # Multi_Head_Attention\'s baseline forward intentionally remains unchanged.\n        # Here, route the same shared program tokens through each token-aware head\n        # and use its existing learned head weights for node and edge aggregation.\n        results = [\n            attentioni(x, program_tokens, program_edge_mask=program_edge_mask)\n            for attentioni in self.attentions.attentions\n        ]\n        node_attention = (\n            torch.stack([result[0] for result in results], dim=-1) @ self.attentions.W_hn\n        ).squeeze(dim=-1)\n        edge_attention = torch.stack([result[1] for result in results], dim=-1) @ self.attentions.W_he\n        edge_attention = edge_attention.squeeze(dim=-1)\n\n        # Combine per-head routing metadata with normalized head weights.  This\n        # yields one program axis K shared by all heads and preserves alpha\'s\n        # interpretation as a neighbor distribution for each receiver/program.\n        head_weights = F.softmax(self.attentions.W_hn.squeeze(dim=-1), dim=0)\n        program_state = {\n            key: torch.sum(\n                torch.stack([result[2][key] for result in results], dim=-1)\n                * head_weights,\n                dim=-1,\n            )\n            for key in ("attention", "messages", "activations")\n        }\n\n        # Pairwise tensors were normalized at entry, so the residual update is\n        # always performed in the explicit ``(B, N, N, E)`` convention.\n        edge = self.ln_edge(edge + edge_attention)\n        node = self.ln1(node + node_attention)\n        node = self.ln2(node + self.FFN(node))\n        return [node, edge, distance, program_state]\n\nclass GRIT_encoder_last_layer(nn.Module):\n    def __init__(self, node_dim, in_node, edge_dim, node_dim_small=16, att_dim=8):\n        super().__init__()\n        self.node_dim_small = node_dim_small\n        self.in_node = in_node\n        self.edge_dim = edge_dim * 2\n        self.att_dim = att_dim\n        self.W_Q = nn.Linear(node_dim, self.edge_dim * att_dim)\n        self.W_K = nn.Linear(node_dim, self.edge_dim * att_dim)\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\n        self.W_En = nn.Linear(edge_dim, node_dim)\n        self.W_A = nn.Linear(edge_dim, in_node, bias=False)\n        self.node_transform = nn.Linear(node_dim, in_node)\n        self.edge_transform = nn.Linear(edge_dim, in_node)\n        self.head = FFN(in_node, in_dim=in_node * 2, out_dim=in_node)\n        self.head2 = nn.Sequential(nn.LayerNorm(in_node), FFN(in_node))\n        self.scaler = 2 / (node_dim ** 0.5)\n\n    def forward(self, x):\n        node, edge, embedding = x\n        B, N, C = node.shape\n        Q = self.W_Q(node[:, 0:1, :]).reshape(B, 1, self.att_dim, self.edge_dim).permute(0, 3, 1, 2)\n        K = self.W_K(node).reshape(B, N, self.att_dim, self.edge_dim).permute(0, 3, 2, 1)\n        QK = (Q @ K).permute(0, 2, 3, 1)\n        edge = edge[:, 0:1, :, :]\n        edge = F.gelu(rho((QK[:, :, :, :self.edge_dim // 2]) * self.W_Ew(edge)) + self.W_Eb(edge) + QK[:, :, :, self.edge_dim // 2:] + embedding[:, 0:1, :, :])\n        edge = edge[:, 0, 1:, :]\n        alphas = F.softmax(self.W_A(edge).permute(0, 2, 1), dim=-1).permute(0, 2, 1)\n        edge = self.edge_transform(edge) * alphas\n        node = self.node_transform(node[:, 1:, :]) * alphas\n        tmp = self.head(torch.concat([edge, node], dim=-1))\n        node = torch.sum(tmp, dim=-2) * self.scaler\n        return [node, [tmp.permute(0, 2, 1).unsqueeze(dim=-2), edge.permute(0, 2, 1).unsqueeze(dim=-2)]]\n\nclass ProgramLastLayer(nn.Module):\n    def __init__(self, node_dim, in_node, edge_dim, n_programs=8, node_dim_small=16, att_dim=8):\n        super().__init__()\n        self.node_dim_small = node_dim_small\n        self.in_node = in_node\n        self.n_programs = n_programs\n        self.edge_dim = edge_dim * 2\n        self.att_dim = att_dim\n        self.W_Q = nn.Linear(node_dim, self.edge_dim * att_dim)\n        self.W_K = nn.Linear(node_dim, self.edge_dim * att_dim)\n        self.W_Ew = nn.Linear(edge_dim, edge_dim, bias=False)\n        self.W_Eb = nn.Linear(edge_dim, edge_dim, bias=False)\n        self.W_A = nn.Linear(edge_dim, in_node * n_programs, bias=False)\n        self.node_transform = nn.Linear(node_dim, in_node)\n        self.edge_transform = nn.Linear(edge_dim, in_node)\n        self.head = FFN(in_node, in_dim=in_node * 2, out_dim=in_node)\n        self.scaler = 2 / (node_dim ** 0.5)\n\n    def forward(self, x):\n        node, edge, embedding = x\n        B, N, C = node.shape\n        Q = self.W_Q(node[:, 0:1, :]).reshape(B, 1, self.att_dim, self.edge_dim).permute(0, 3, 1, 2)\n        K_mat = self.W_K(node).reshape(B, N, self.att_dim, self.edge_dim).permute(0, 3, 2, 1)\n        QK = (Q @ K_mat).permute(0, 2, 3, 1)\n        edge = edge[:, 0:1, :, :]\n        edge = F.gelu(\n            rho((QK[:, :, :, :self.edge_dim // 2]) * self.W_Ew(edge))\n            + self.W_Eb(edge)\n            + QK[:, :, :, self.edge_dim // 2:]\n            + embedding[:, 0:1, :, :]\n        )\n        edge = edge[:, 0, 1:, :]\n        alphas_all = self.W_A(edge)\n        alphas_all = alphas_all.reshape(B, N - 1, self.in_node, self.n_programs)\n        alphas_all = alphas_all.permute(0, 3, 2, 1)\n        alphas_all = F.softmax(alphas_all, dim=-1)\n        alphas_all = alphas_all.permute(0, 3, 1, 2)\n        edge_feat = self.edge_transform(edge)\n        node_feat = self.node_transform(node[:, 1:, :])\n        program_outputs = []\n        program_influences = []\n        for k in range(self.n_programs):\n            alpha_k = alphas_all[:, :, k, :]\n            edge_k = edge_feat * alpha_k\n            node_k = node_feat * alpha_k\n            tmp_k = self.head(torch.cat([edge_k, node_k], dim=-1))\n            output_k = torch.sum(tmp_k, dim=1) * self.scaler\n            program_outputs.append(output_k)\n            program_influences.append(tmp_k)\n        y_pred = torch.stack(program_outputs, dim=0).sum(dim=0)\n        total_influence = torch.stack(program_influences, dim=0).sum(dim=0)\n        influence_for_compat = total_influence.permute(0, 2, 1).unsqueeze(dim=-2)\n        program_info = {\n            \'per_program_influence\': torch.stack(program_influences, dim=0),\n            \'per_program_output\': torch.stack(program_outputs, dim=0),\n            \'per_program_attention\': alphas_all,\n        }\n        return [y_pred, [influence_for_compat, program_info]]\n', 'spagat_r15/dataloader.py': 'import torch\nfrom torch.utils.data import Dataset\nimport os\nimport numpy as np\nimport pandas as pd\n\nclass SPAGAT_dataset(Dataset):\n    def __init__(self, processed_dir, num_neighbors=50, col_centerx="centerx",\n                 col_centery="centery", col_cell_type="subclass"):\n        if processed_dir[-1] != "/":\n            processed_dir = processed_dir + "/"\n\n        samples = []\n        for filei in os.listdir(processed_dir):\n            if filei.find("_TypeExp.npz") >= 0:\n                samples.append(filei.split("_TypeExp.npz")[0])\n        self.samples = list(sorted(list(set(samples))))\n        print("Have samples:", len(self.samples), self.samples)\n\n        self.index_index = ["index_" + str(i) for i in range(num_neighbors)]\n        self.interactions = torch.load("/".join(processed_dir.split("/")[:-2]) + "/ligands.pth", weights_only=False)\n        self.genes = torch.load("/".join(processed_dir.split("/")[:-2]) + "/genes.pth", weights_only=False)\n\n        # Collect all cell types from all data files\n        all_cell_types = set()\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                all_cell_types.update(dfi[col_cell_type].unique())\n            except Exception as e:\n                print(f"Warning: Cannot read sample {samplei}: {e}")\n                continue\n        # Build cell type dictionary, sorted for consistency\n        self.cell_types_dict = {ct: i for i, ct in enumerate(sorted(all_cell_types))}\n        print(f"Built cell type dictionary with {len(self.cell_types_dict)} cell types: {sorted(all_cell_types)}")\n\n        self.indexes = []\n        self.flags = []\n        self.centerx = []\n        self.centery = []\n        self.exps = []\n        self.cell_types = []\n        self.type_exp = []\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                # Assign cell type numbers for each cell in the sample\n                dfi["cell_type_number"] = np.array([self.cell_types_dict.get(cell_type, -1)\n                                                     for cell_type in dfi[col_cell_type]])\n                if "flag" not in dfi.columns.tolist():\n                    dfi["flag"] = [True for _ in range(dfi.shape[0])]\n\n                type_exp_dicti = np.load(processed_dir + samplei + "_TypeExp.npz", allow_pickle=True)\n                type_exp_dicti = {k: torch.Tensor(np.array(v).astype(np.float64)) for k, v in type_exp_dicti.items()}\n                # Only keep cell types present in both the dict and the global cell type dict\n                valid_cell_types = [ct for ct in dfi[col_cell_type].unique()\n                                   if ct in type_exp_dicti and ct in self.cell_types_dict]\n                type_exp = torch.stack([type_exp_dicti[cell_type] for cell_type in dfi[col_cell_type]\n                                       if cell_type in valid_cell_types], dim=0)\n                # Skip sample if no valid cell types\n                if len(valid_cell_types) == 0:\n                    print(f"Skip sample {samplei}: no valid cell types")\n                    continue\n                # Only keep rows with valid cell types\n                valid_mask = dfi[col_cell_type].isin(valid_cell_types)\n                dfi_valid = dfi[valid_mask].copy()\n                if len(dfi_valid) == 0:\n                    print(f"Skip sample {samplei}: no valid rows after filtering")\n                    continue\n                self.indexes.append(torch.LongTensor(dfi_valid.loc[:, self.index_index].values))\n                self.flags.append(dfi_valid.loc[:, "flag"].values)\n                self.centerx.append(torch.Tensor(dfi_valid.loc[:, col_centerx].values))\n                self.centery.append(torch.Tensor(dfi_valid.loc[:, col_centery].values))\n                self.exps.append(torch.Tensor(dfi_valid.loc[:, self.genes].values))\n                self.cell_types.append(torch.LongTensor(dfi_valid.loc[:, "cell_type_number"].values))\n                # Rebuild type_exp for valid cell types\n                type_exp_valid = torch.stack([type_exp_dicti[cell_type] for cell_type in dfi_valid[col_cell_type]], dim=0)\n                self.type_exp.append(type_exp_valid)\n            except Exception as e:\n                print(f"Error: Cannot process sample {samplei}: {e}")\n                import traceback\n                traceback.print_exc()\n                continue\n\n        self.meta_counts = []\n        self.arg_meta = []\n        for i in range(len(self.samples)):\n            if i < len(self.flags):\n                self.meta_counts.append(np.sum(self.flags[i]))\n                self.arg_meta.append(torch.LongTensor(np.where(self.flags[i] != 0)[0]))\n        self.meta_counts = np.array(self.meta_counts) if self.meta_counts else np.array([])\n        self.cumsum_meta = np.cumsum(self.meta_counts) if len(self.meta_counts) > 0 else np.array([])\n        print("There are totally", self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0, "cells in this dataset")\n\n    def __len__(self):\n        return self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0\n\n    def __getitem__(self, idx):\n        if len(self.cumsum_meta) == 0:\n            raise RuntimeError("No valid data in dataset")\n        # Find which sample this idx comes from\n        sample_id = np.searchsorted(self.cumsum_meta, idx, side=\'right\')\n        idx = idx - self.cumsum_meta[sample_id]\n        idx = self.arg_meta[sample_id][idx]\n        indices = self.indexes[sample_id][idx]\n        centerx = self.centerx[sample_id][indices]\n        centery = self.centery[sample_id][indices]\n        exp = self.exps[sample_id][indices]\n        type_exp = self.type_exp[sample_id][indices]\n        y = exp[0]\n        cell_types = torch.LongTensor(self.cell_types[sample_id][indices])\n        num_real_neighbors = len(indices)\n        neighbor_mask = torch.ones(num_real_neighbors)\n        return {\n            "x": exp,\n            "type_exp": type_exp,\n            "y": y,\n            "cell_types": cell_types,\n            "position_x": centerx,\n            "position_y": centery,\n            "neighbor_mask": neighbor_mask\n        }\n\nclass SPAGAT_evaluate_dataset(Dataset):\n    def __init__(self, processed_dir, sample, num_neighbors=50, col_centerx="centerx",\n                 col_centery="centery", col_cell_type="subclass"):\n        if processed_dir[-1] != "/":\n            processed_dir = processed_dir + "/"\n        self.samples = [sample]\n        print("Have samples:", self.samples)\n        self.index_index = ["index_" + str(i) for i in range(num_neighbors)]\n        self.interactions = torch.load("/".join(processed_dir.split("/")[:-2]) + "/ligands.pth", weights_only=False)\n        self.genes = torch.load("/".join(processed_dir.split("/")[:-2]) + "/genes.pth", weights_only=False)\n        # Collect all cell types from all data files\n        all_cell_types = set()\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                all_cell_types.update(dfi[col_cell_type].unique())\n            except Exception as e:\n                print(f"Warning: Cannot read sample {samplei}: {e}")\n                continue\n        # Build cell type dictionary, sorted for consistency\n        self.cell_types_dict = {ct: i for i, ct in enumerate(sorted(all_cell_types))}\n        print(f"Built cell type dictionary with {len(self.cell_types_dict)} cell types: {sorted(all_cell_types)}")\n        self.indexes = []\n        self.flags = []\n        self.centerx = []\n        self.centery = []\n        self.exps = []\n        self.cell_types = []\n        self.type_exp = []\n        for samplei in self.samples:\n            try:\n                dfi = pd.read_csv(processed_dir + samplei + ".csv")\n                dfi["cell_type_number"] = np.array([self.cell_types_dict.get(cell_type, -1)\n                                                     for cell_type in dfi[col_cell_type]])\n                if "flag" not in dfi.columns.tolist():\n                    dfi["flag"] = [True for _ in range(dfi.shape[0])]\n                type_exp_dicti = np.load(processed_dir + samplei + "_TypeExp.npz", allow_pickle=True)\n                type_exp_dicti = {k: torch.Tensor(np.array(v).astype(np.float64)) for k, v in type_exp_dicti.items()}\n                type_exp = torch.stack([type_exp_dicti[cell_type] for cell_type in dfi[col_cell_type]], dim=0)\n                self.type_exp.append(type_exp)\n                self.indexes.append(torch.LongTensor(dfi.loc[:, self.index_index].values))\n                self.flags.append(dfi.loc[:, "flag"].values)\n                self.centerx.append(torch.Tensor(dfi.loc[:, col_centerx].values))\n                self.centery.append(torch.Tensor(dfi.loc[:, col_centery].values))\n                self.exps.append(torch.Tensor(dfi.loc[:, self.genes].values))\n                self.cell_types.append(torch.LongTensor(dfi.loc[:, "cell_type_number"].values))\n            except Exception as e:\n                print(f"Error: Cannot process sample {samplei}: {e}")\n                import traceback\n                traceback.print_exc()\n                continue\n        self.meta_counts = []\n        self.arg_meta = []\n        for i in range(len(self.samples)):\n            if i < len(self.flags):\n                self.meta_counts.append(np.sum(self.flags[i]))\n                self.arg_meta.append(torch.LongTensor(np.where(self.flags[i] != 0)[0]))\n        self.meta_counts = np.array(self.meta_counts) if self.meta_counts else np.array([])\n        self.cumsum_meta = np.cumsum(self.meta_counts) if len(self.meta_counts) > 0 else np.array([])\n        print("There are totally", self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0, "cells in this dataset")\n\n    def __len__(self):\n        return self.cumsum_meta[-1] if len(self.cumsum_meta) > 0 else 0\n\n    def __getitem__(self, idx):\n        if len(self.cumsum_meta) == 0:\n            raise RuntimeError("No valid data in dataset")\n        sample_id = np.searchsorted(self.cumsum_meta, idx, side=\'right\')\n        idx = idx - self.cumsum_meta[sample_id]\n        idx = self.arg_meta[sample_id][idx]\n        indices = self.indexes[sample_id][idx]\n        centerx = self.centerx[sample_id][indices]\n        centery = self.centery[sample_id][indices]\n        exp = self.exps[sample_id][indices]\n        type_exp = self.type_exp[sample_id][indices]\n        y = exp[0]\n        cell_types = torch.LongTensor(self.cell_types[sample_id][indices])\n        indices = torch.LongTensor(indices)\n        return {\n            "x": exp,\n            "type_exp": type_exp,\n            "y": y,\n            "cell_types": cell_types,\n            "position_x": centerx,\n            "position_y": centery,\n            "NN": indices\n        }\n', 'spagat_r15/embedding.py': 'import torch\nimport torch.nn as nn\nimport torch.nn.functional as F\nfrom . import preprocess\n\nclass FFN(nn.Module):\n    def __init__(self,dim,in_dim=None,out_dim=None,bias=True):\n        super().__init__()\n        if in_dim is None:\n            in_dim=dim\n        if out_dim is None:\n            out_dim=dim\n\n        self.linear1=nn.Linear(in_dim,dim*4,bias=bias)\n        self.linear2=nn.Linear(dim*4,out_dim,bias=bias)\n        self.act=nn.GELU()\n\n    def forward(self,x):\n        return self.linear2(self.act(self.linear1(x)))\n\ndef group_strings_by_numbers(strings, numbers):\n    """\n    Group a list of strings based on unique numbers and return as two ordered lists.\n\n    :param strings: List of strings to be grouped.\n    :param numbers: List of numbers indicating the grouping.\n    :return: Two lists, one with unique numbers and the other with corresponding grouped strings.\n    """\n    numbers=numbers.copy()[:len(strings)]\n    unique_numbers = sorted(set(numbers))\n    grouped_strings = [[] for _ in unique_numbers]\n\n    for string, number in zip(strings, numbers):\n        index = unique_numbers.index(number)\n        grouped_strings[index].append(string)\n\n    return unique_numbers, grouped_strings\n\nclass Embedding(nn.Module):\n    def __init__(self,genes,ligands_info,node_dim,edge_dim,use_cell_type_embedding=True):\n        super().__init__()\n        self.use_cell_type_embedding=use_cell_type_embedding\n        self.ligands_info=ligands_info\n\n        # for scaling the data\n        #self.exp_scaler=nn.Parameter(torch.ones((len(genes))))\n\n        # for node\n        self.cell_encoder1 = FFN(node_dim, in_dim=len(genes))\n        self.cell_encoder2 = nn.Embedding(29,node_dim)\n\n        # for distance\n        self.distance_scaler=nn.Sequential(FFN(edge_dim,out_dim=len(ligands_info[0]),in_dim=5),nn.Sigmoid())\n        self.distance_encoder=FFN(edge_dim,out_dim=len(ligands_info[0]),in_dim=5)\n        self.distance_embedding = nn.Sequential(FFN(edge_dim, in_dim=5), nn.Sigmoid())\n\n        # for edge\n        self.ligands_encoder=nn.Linear(len(ligands_info[0]),edge_dim)\n        self.edge_encoder=nn.Linear(len(ligands_info[0])+edge_dim,edge_dim,bias=False)\n        self.scaler=edge_dim**0.5\n\n        # calculate ligand scores\n        self.ligands_index=[]\n        for i in range(len(self.ligands_info[0])):\n            ligandi = self.ligands_info[0][i]\n            ligand_groupi = self.ligands_info[1][i]\n            uniquei, groupi = group_strings_by_numbers(ligandi, ligand_groupi)\n            groupi=[torch.LongTensor([genes.index(groupi[j][k]) for k in range(len(groupi[j]))]) for j in range(len(groupi))]\n            self.ligands_index.append(groupi)\n\n    def forward(self, x, ligand_feature_mask=None):\n        \'\'\'\n        :param x:\n        x["x"]--b,n,in_node\n        x["distance_matrix"]--b,5,n,n\n        x["cell_types"]--long,b,n\n        x["type_exp"]--b,n,in_node\n        :return:\n        node_features:b,n,node_dim\n        edge1:b,n,n,edge_dim\n        edge2:b,n,n,edge_dim+5\n        rw: b,n,n,num_hops+5\n        \'\'\'\n        x=preprocess.process(x)\n\n        # node features\n        #scaler=torch.square(self.exp_scaler)\n        exp=x["type_exp"]+x["x"]\n        exp=torch.where(x["cell_types"].unsqueeze(dim=-1)==x["cell_types"][:,0:1].unsqueeze(dim=-1),x["type_exp"],exp)\n        exp=exp#*scaler\n        node_features1=self.cell_encoder1(exp)\n        node_features2 = self.cell_encoder1(x["type_exp"])#*scaler)\n        node_features=torch.where(x["cell_types"].unsqueeze(dim=-1)==x["cell_types"][:,0:1].unsqueeze(dim=-1),node_features2,node_features1)\n\n        if self.use_cell_type_embedding:\n            node_features=node_features+self.cell_encoder2(x["cell_types"])\n\n        # calculate ligand scores\n        B,N,C=node_features.shape\n        data_ligand=[]\n        for i in range(len(self.ligands_info[0])):\n            data_ligandi=torch.ones((B,N),device=node_features.device)\n            for j in range(len(self.ligands_index[i])):\n                data_ligandi = data_ligandi * torch.mean(exp[:,:,self.ligands_index[i][j]],dim=-1)\n            data_ligandi=torch.pow(data_ligandi,exponent=1 / len(self.ligands_index[i]))\n            data_ligand.append(data_ligandi)\n        data_ligand=torch.stack(data_ligand,dim=-1)\n\n        # distance scaler\n        distance_scaler = self.distance_scaler(x["distance_matrix"])\n\n        # ligand score scaled by distance\n        ligand_score=distance_scaler*data_ligand\n        if ligand_feature_mask is not None:\n            ligand_score = ligand_score * ligand_feature_mask\n        edge_ligand=self.ligands_encoder(ligand_score)\n\n        # edge features\n        edge_distance=self.distance_encoder(x["distance_matrix"])\n        edges = self.edge_encoder(torch.concat([edge_distance,edge_ligand],dim=-1))*self.scaler\n\n        # distance embedding\n        embedding = self.distance_embedding(x["distance_matrix"])\n        return [node_features,edges.unsqueeze(dim=1),embedding.unsqueeze(dim=1)]\n\n\n\n', 'spagat_r15/gene_program_model.py': '"""\nSpatial Gene Program Model (SpaGP)\n\nCore idea:\n    y_i = μ(cell_type_i) + Σ_k g_k(z_i) · p_k + ε\n\nwhere:\n    - μ(cell_type) is cell-type baseline expression (from type_exp)\n    - p_k ∈ R^genes is the k-th gene program (global, shared, orthogonal)\n    - g_k(z_i) ∈ R+ is scalar activation of program k for cell i (non-negative)\n    - z_i ∈ R^d is spatial context embedding from neighborhood via graph encoder\n"""\n\nimport torch\nimport torch.nn as nn\nimport torch.nn.functional as F\n\nfrom .attention import GRIT_encoder\nfrom .embedding import Embedding, FFN\n\n\nclass GeneProgramHead(nn.Module):\n    """\n    The program-based prediction head.\n    \n    Given spatial context embedding z_i (from graph encoder),\n    predicts gene expression as:\n        y_i = baseline_i + Σ_k g_k(z_i) · p_k\n    """\n    def __init__(self, node_dim, n_genes, n_programs=16, n_neighbors=50,\n                 program_aware=False, message_decoder="free"):\n        super().__init__()\n        self.n_genes = n_genes\n        self.n_programs = n_programs\n        self.n_neighbors = n_neighbors\n        self.program_aware = program_aware\n        self.message_decoder = message_decoder\n        if message_decoder not in {"free", "program_aligned"}:\n            raise ValueError("message_decoder must be \'free\' or \'program_aligned\'.")\n        \n        # ===== Gene Programs p_k ∈ R^{K × genes} =====\n        # Initialized with Xavier, will be orthogonalized during training via loss penalty\n        self.programs = nn.Parameter(torch.empty(n_programs, n_genes))\n        nn.init.xavier_uniform_(self.programs)\n        \n        # ===== Context Aggregator: aggregate neighborhood into z_i =====\n        # Attention-weighted aggregation of neighbor node features → z_i\n        self.context_query = nn.Linear(node_dim, node_dim)\n        self.context_key = nn.Linear(node_dim, node_dim)\n        self.context_proj = nn.Sequential(\n            nn.Linear(node_dim, node_dim),\n            nn.GELU(),\n            nn.Linear(node_dim, node_dim)\n        )\n        self.scale = node_dim ** 0.5\n        \n        # ===== Activation Network: z_i → g(z_i) ∈ R+^K =====\n        self.activation_net = nn.Sequential(\n            nn.Linear(node_dim, node_dim),\n            nn.GELU(),\n            nn.Linear(node_dim, n_programs),\n            nn.Softplus()  # ensures g_k ≥ 0\n        )\n        \n        # ===== Baseline: cell-type expression → gene-level baseline =====\n        self.baseline_head = nn.Sequential(\n            nn.Linear(n_genes, n_genes),\n            nn.GELU(),\n            nn.Linear(n_genes, n_genes)\n        )\n\n        # Only program-aware SpaGP uses this decoder branch.  Keeping it absent\n        # from the legacy construction preserves existing checkpoint layouts.\n        if program_aware:\n            if message_decoder == "free":\n                self.message_to_genes = nn.Linear(node_dim, n_genes)\n            else:\n                # A shared scalar decoder yields q_k from each program message\n                # m_k.  Its gene effect is constrained to that program\'s P_k.\n                self.message_to_program = nn.Linear(node_dim, 1)\n    \n    def forward(self, node_features, type_exp_center, encoder_program_state=None, program_mask=None):\n        """\n        Args:\n            node_features: (B, N, node_dim) — encoder output for center + neighbors\n            type_exp_center: (B, n_genes) — cell-type average expression for center cell\n            encoder_program_state: Optional state returned by a program-aware\n                GRIT encoder.  When supplied, its center-cell activations replace\n                this head\'s legacy context/activation inference.\n            program_mask: Optional inference-only ``(K,)`` 0/1 mask. It is\n                applied only to program-specific decoder contributions; the\n                shared cell-type baseline remains unchanged.\n        \n        Returns:\n            y_pred: (B, n_genes)\n            program_info: dict with activations and programs for downstream analysis\n        """\n        B, N, C = node_features.shape\n        \n        if encoder_program_state is None:\n            # Legacy decoder path: retain the original head computation and\n            # parameter layout so existing SpaGP checkpoints remain usable.\n            center = node_features[:, 0:1, :]\n            neighbors = node_features[:, 1:, :]\n            Q = self.context_query(center)\n            K = self.context_key(neighbors)\n            attn = F.softmax((Q @ K.transpose(-1, -2)) / self.scale, dim=-1)\n            z_i = self.context_proj(attn @ neighbors).squeeze(1)\n            g = self.activation_net(z_i)\n            program_attention = None\n            program_messages = None\n        else:\n            # Program-aware decoder path: graph message passing has already\n            # produced non-negative activations for every node/program pair.\n            # The center-cell row is the activation vector used to decode genes.\n            g = encoder_program_state[\'activations\'][:, 0, :]\n            z_i = node_features[:, 0, :]\n            program_attention = encoder_program_state[\'attention\'][:, 0, 1:, :]\n            program_messages = encoder_program_state[\'messages\'][:, 0, :, :]\n\n            # Keep the historical (B, N-1) attn_weights field as an activation-\n            # weighted summary of the full program-specific attention tensor.\n            gate_weights = g / (torch.sum(g, dim=-1, keepdim=True) + 1e-8)\n            attn = torch.sum(program_attention * gate_weights.unsqueeze(dim=1), dim=-1).unsqueeze(dim=1)\n        \n        # --- Step 3: Normalize programs (unit norm per program) ---\n        programs_normalized = F.normalize(self.programs, dim=-1)  # (K, genes)\n        if program_mask is not None:\n            program_mask = torch.as_tensor(\n                program_mask, device=programs_normalized.device, dtype=programs_normalized.dtype\n            )\n            if program_mask.ndim != 1 or program_mask.shape[0] != self.n_programs:\n                raise ValueError(\n                    f"program_mask must have shape ({self.n_programs},); got {tuple(program_mask.shape)}."\n                )\n        mask_is_identity = program_mask is None or bool(torch.all(program_mask == 1))\n        \n        # --- Step 4: Program contribution ---\n        # Σ_k g_k(z_i) · p_k = g @ P\n        # Keep the default expression byte-for-byte on the existing path.  For\n        # an intervention, mask each program\'s gene-program contribution before\n        # summing across k; no re-normalization occurs.\n        masked_g = g if mask_is_identity else g * program_mask.unsqueeze(0)\n        program_output = masked_g @ programs_normalized   # (B, genes)\n\n        if encoder_program_state is None:\n            message_output = None\n            message_coefficients = None\n        else:\n            if not self.program_aware:\n                raise RuntimeError(\n                    "GeneProgramHead received encoder program state but was not "\n                    "constructed with program_aware=True."\n                )\n            if self.message_decoder == "free":\n                # Legacy program-aware ablation: a flexible shared gene decoder\n                # consumes the gate-weighted sum of program messages.\n                if mask_is_identity:\n                    fused_program_message = torch.sum(\n                        program_messages * gate_weights.unsqueeze(dim=-1), dim=1\n                    )\n                    message_output = self.message_to_genes(fused_program_message)\n                else:\n                    # Distribute the free decoder\'s shared bias by the original\n                    # program weights: sum_k w_k (W m_k + b) equals the legacy\n                    # W sum_k(w_k m_k) + b when every mask is one, while an\n                    # all-zero mask removes the complete program message path.\n                    message_weights = gate_weights * program_mask.unsqueeze(0)\n                    message_components = F.linear(\n                        program_messages, self.message_to_genes.weight, self.message_to_genes.bias\n                    )\n                    message_output = torch.sum(message_components * message_weights.unsqueeze(dim=-1), dim=1)\n                message_coefficients = None\n            else:\n                # Program-aligned decoder: q_k is derived from m_k, then its\n                # contribution is restricted to the corresponding gene program\n                # P_k rather than passing through an unrestricted gene decoder.\n                message_coefficients = self.message_to_program(program_messages).squeeze(dim=-1)\n                masked_coefficients = (\n                    message_coefficients if mask_is_identity\n                    else message_coefficients * program_mask.unsqueeze(0)\n                )\n                message_output = masked_coefficients @ programs_normalized\n        \n        # --- Step 5: Baseline from cell-type expression ---\n        baseline = self.baseline_head(type_exp_center)  # (B, genes)\n        \n        # --- Step 6: Final prediction ---\n        y_pred = baseline + program_output\n        if message_output is not None:\n            y_pred = y_pred + message_output\n        \n        program_info = {\n            \'activations\': g,                    # (B, K) — program activation per cell\n            \'programs\': programs_normalized,      # (K, genes) — gene program vectors\n            \'context\': z_i,                       # (B, C) — spatial context embedding\n            \'attn_weights\': attn.squeeze(1),      # (B, N-1) — neighbor attention weights\n            \'baseline\': baseline,                 # (B, genes) — cell-type baseline\n            \'program_output\': program_output      # (B, genes) — program contribution\n        }\n        if program_attention is not None:\n            program_info[\'program_attention\'] = program_attention\n            program_info[\'program_messages\'] = program_messages\n            program_info[\'program_message_output\'] = message_output\n            program_info[\'program_message_coefficients\'] = message_coefficients\n        if program_mask is not None:\n            program_info[\'program_mask\'] = program_mask\n        \n        return y_pred, program_info\n\n\nclass SpaGP(nn.Module):\n    """\n    Spatial Gene Program Model.\n    """\n    def __init__(self, genes, ligands_info, node_dim=256, edge_dim=48, num_heads=2,\n                 n_layers=1, att_dim=8, use_cell_type_embedding=True,\n                 n_programs=16, program_aware=False, program_token_dim=32,\n                 message_decoder="free", routing_mode="program"):\n        super().__init__()\n        self.genes = genes\n        self.n_programs = n_programs\n        self.program_aware = program_aware\n        self.program_token_dim = program_token_dim\n        self.message_decoder = message_decoder\n        self.routing_mode = routing_mode\n\n        if program_aware and n_layers < 1:\n            raise ValueError("SpaGP program-aware mode requires at least one GRIT encoder layer.")\n        \n        self.embeddings = Embedding(genes, ligands_info, node_dim, edge_dim,\n                                    use_cell_type_embedding=use_cell_type_embedding)\n        self.encoders = nn.ModuleList([\n            GRIT_encoder(node_dim, edge_dim, num_heads, att_dim, program_aware=program_aware, routing_mode=routing_mode)\n            for i in range(n_layers)  \n        ])\n\n        if program_aware:\n            # The learned bank is intentionally in a separate latent space P.\n            # Its shared projection supplies (K, edge_dim) tokens required by\n            # every GRIT layer, keeping program identity fixed across depth.\n            self.program_tokens = nn.Parameter(torch.empty(n_programs, program_token_dim))\n            nn.init.xavier_uniform_(self.program_tokens)\n            self.program_token_projection = nn.Linear(program_token_dim, edge_dim, bias=False)\n\n        self.program_head = GeneProgramHead(\n            node_dim=node_dim,\n            n_genes=len(genes),\n            n_programs=n_programs,\n            program_aware=program_aware,\n            message_decoder=message_decoder,\n        )\n    \n    def forward(self, x, program_mask=None, program_edge_mask=None,ligand_feature_mask=None):\n        """\n        Args:\n            x: dict from dataloader with keys:\n                "x": (B, N, genes), "type_exp": (B, N, genes),\n                "cell_types": (B, N), "position_x": (B, N), "position_y": (B, N),\n                "y": (B, genes)\n        \n        Returns:\n            y_pred: (B, genes)\n            program_info: dict with activations, programs, context, etc.\n        """\n        # Center cell\'s type expression (for baseline)\n        type_exp_center = x["type_exp"][:, 0, :]  # (B, genes)\n        encoded = self.embeddings(x,ligand_feature_mask=ligand_feature_mask)   # [node, edge, distance]\n        if self.program_aware:\n            program_tokens = self.program_token_projection(self.program_tokens)\n            final_program_state = None\n            for encoder in self.encoders:\n                # Program state is analysis metadata; only graph tensors are\n                # forwarded to the next GRIT layer.\n                encoded = encoder(\n                    encoded[:3], program_tokens=program_tokens, program_edge_mask=program_edge_mask\n                )\n                final_program_state = encoded[3]\n        else:\n            for encoder in self.encoders:\n                encoded = encoder(encoded)\n            final_program_state = None\n        node_features = encoded[0]     # (B, N, node_dim)\n        \n        # Program-based prediction\n        y_pred, program_info = self.program_head(\n            node_features, type_exp_center, encoder_program_state=final_program_state,\n            program_mask=program_mask,\n        )\n        \n        return y_pred, program_info\n\n\nclass SpaGP_Loss(nn.Module):\n    """\n    Loss function for SpaGP model:\n        L = L_mse + λ_orth * L_orthogonality + λ_sparse * L_sparsity\n    """\n    def __init__(self, gene_list, interaction_gene_list1, \n                 lambda_orth=0.1, lambda_sparse=0.01):\n        super().__init__()\n        interaction_gene_list = list(set(elem for sublist in interaction_gene_list1[0] for elem in sublist))\n        \n        self.interaction_gene_index = []\n        self.not_interaction_gene_index = []\n        for i in range(len(gene_list)):\n            if gene_list[i] in interaction_gene_list:\n                self.interaction_gene_index.append(i)\n            else:\n                self.not_interaction_gene_index.append(i)\n        self.interaction_gene_index = torch.LongTensor(self.interaction_gene_index)\n        self.not_interaction_gene_index = torch.LongTensor(self.not_interaction_gene_index)\n        \n        self.mse = nn.MSELoss()\n        self.lambda_orth = lambda_orth\n        self.lambda_sparse = lambda_sparse\n    \n    def orthogonality_loss(self, programs):\n        """\n        Soft orthogonality penalty: Σ_{i≠j} (p_i · p_j)²\n        \n        Args:\n            programs: (K, genes) — normalized program vectors\n        """\n        # Gram matrix: (K, K)\n        gram = programs @ programs.T  # p_i · p_j for all pairs\n        # Zero out diagonal (self-similarity = 1, we don\'t penalize that)\n        K = programs.shape[0]\n        if K < 2:\n            # A single program has no program pair, so orthogonality/diversity\n            # is defined exactly as zero rather than evaluating 0 / (K * (K-1)).\n            return programs.new_zeros(())\n        mask = 1.0 - torch.eye(K, device=programs.device)\n        off_diag = gram * mask\n        # Penalty: sum of squared off-diagonal entries\n        return torch.sum(off_diag ** 2) / (K * (K - 1))\n    \n    def sparsity_loss(self, activations):\n        """\n        L1 sparsity on program activations.\n        Encourages each cell to activate only a few programs.\n        \n        Args:\n            activations: (B, K) — non-negative program activations\n        """\n        return torch.mean(activations)  # L1 of non-negative = mean\n    \n    def forward(self, y_pred_tuple, y):\n        """\n        Args:\n            y_pred_tuple: (y_pred, program_info) from model\n            y: (B, genes) ground truth\n        \n        Returns:\n            total_loss: scalar (for backward)\n            mse_not_interact: scalar (for monitoring, detached)\n            orth_loss: scalar (for monitoring, detached)\n            sparse_loss: scalar (for monitoring, detached)\n        """\n        y_pred, program_info = y_pred_tuple\n        \n        # Main prediction loss\n        mse_loss = self.mse(y_pred, y)\n        \n        # Orthogonality penalty\n        orth_loss = self.orthogonality_loss(program_info[\'programs\'])\n        \n        # Sparsity penalty\n        sparse_loss = self.sparsity_loss(program_info[\'activations\'])\n        \n        # Total\n        total_loss = mse_loss + self.lambda_orth * orth_loss + self.lambda_sparse * sparse_loss\n        \n        # Monitoring: MSE on non-interaction genes\n        mse_not_interact = self.mse(\n            y_pred[:, self.not_interaction_gene_index],\n            y[:, self.not_interaction_gene_index]\n        ).detach().cpu()\n        \n        return (total_loss, mse_not_interact, \n                orth_loss.detach().cpu(), sparse_loss.detach().cpu())\nclass SpaGP_NoSpatial(nn.Module):\n    """\n    Ablation model: g_k predicted from center cell expression only.\n    No neighborhood information → if VE ≈ full model, spatial story fails.\n    """\n    def __init__(self, genes, ligands_info, node_dim=256, edge_dim=48, num_heads=2,\n                 n_layers=1, att_dim=8, use_cell_type_embedding=True,\n                 n_programs=16):\n        super().__init__()\n        self.genes = genes\n        self.n_programs = n_programs\n        n_genes = len(genes)\n        \n        # Programs (same as SpaGP)\n        self.programs = nn.Parameter(torch.empty(n_programs, n_genes))\n        nn.init.xavier_uniform_(self.programs)\n        \n        # Activation from center cell expression ONLY (no spatial context)\n        self.activation_net = nn.Sequential(\n            nn.Linear(n_genes, 128),\n            nn.GELU(),\n            nn.Linear(128, n_programs),\n            nn.Softplus()\n        )\n    \n    def forward(self, x):\n        # Only use center cell\'s raw expression — NO neighbors\n        center_exp = x["type_exp"][:, 0, :] + x["x"][:, 0, :]  # (B, genes)\n        \n        g = self.activation_net(center_exp)  # (B, K)\n        programs_norm = F.normalize(self.programs, dim=-1)\n        y_pred = g @ programs_norm\n        \n        program_info = {\n            \'activations\': g,\n            \'programs\': programs_norm,\n            \'context\': center_exp,\n        }\n        return y_pred, program_info\n', 'spagat_r15/preprocess.py': 'import torch\n\ndef process(x,judge_distance=28):\n    \'\'\'\n    :param x:\n        "x": exp,\n        "y": y,\n        "ligand":ligand,\n        "receptor":receptor,\n        "cell_types": cell_types,\n        "morphology": volumes,\n        "position_x": centerx,\n        "position_y": centery\n    :return:\n        "x": b,n,c1\n        "interactions": b,n,n,c2\n        "mask": b,n,n\n        "cell_types": b,n\n        "distance_matrix": b,n,n,c3\n        "y": b,c\n        "morphology": b,n,c4\n    \'\'\'\n    dx = x["position_x"][:, 0:1] - x["position_x"]\n    dy = x["position_y"][:, 0:1] - x["position_y"]\n    distances=torch.sqrt(torch.square(dx)+torch.square(dy))\n\n    \'\'\'mask=torch.where(distances<judge_distance,\n                     torch.ones_like(distances,device=distances.device),\n                     torch.zeros_like(distances,device=distances.device))\'\'\'\n    distance_matrix = torch.stack([1 / (distances + 1), 1 / (torch.sqrt(distances) + 1),\n                                   1 / (1 + torch.square(distances)), torch.exp(-distances),\n                                   torch.exp(-torch.square(distances))], dim=-1)\n    return {\n        "x":x["x"],\n        "type_exp": x["type_exp"],\n        "y":x["y"],\n        "cell_types": x["cell_types"],\n        "distance_matrix":distance_matrix\n    }', 'r15_core.py': '"""Distance-stratified mask selection. No training or outcome-dependent selection."""\nimport numpy as np\n\nBIN_WIDTH = 0.05\nN_BINS = 20\nREPEATS = 10\nMASK_SEED = 123\nPOLICIES = (\'count_only_same_m\', \'count_distance_bins\')\nCONDITIONS = (\'tumor_1\', \'tumor_2\', \'random\')\nOPERATORS = (\'edge_exclusion\', \'fixed_weight_messages\')\n\n\ndef original_batch_selection(n_original, eligible_positions, batch_start, batch_size):\n    """Map a full, original export batch to positions in the eligible cohort."""\n    positions = np.asarray(eligible_positions, dtype=np.int64)\n    end = min(batch_start + batch_size, n_original)\n    lo, hi = np.searchsorted(positions, [batch_start, end])\n    local = positions[lo:hi] - batch_start\n    return np.arange(lo, hi, dtype=np.int64), local\n\n\ndef discrepancy_report(actual, expected, atol, rtol):\n    """Report discrepancies without changing the original acceptance threshold."""\n    actual, expected = np.asarray(actual), np.asarray(expected)\n    difference = np.abs(actual.astype(np.float64) - expected.astype(np.float64))\n    close = np.isclose(actual, expected, atol=atol, rtol=rtol, equal_nan=False)\n    return dict(passed=bool(close.all()), n_failing_values=int((~close).sum()),\n                n_failing_rows=int((~close).any(axis=1).sum()),\n                maximum_absolute_difference=float(difference.max()),\n                mean_absolute_difference=float(difference.mean()),\n                atol=atol, rtol=rtol), ~close\n\n\ndef distance_plan(types, px, py, sender_ids):\n    """All 50 positions include center 0. Bins are fixed before seeing predictions."""\n    types = np.asarray(types)\n    px, py = np.asarray(px, dtype=np.float64), np.asarray(py, dtype=np.float64)\n    if types.ndim != 2 or types.shape[1] != 50 or px.shape != types.shape or py.shape != types.shape:\n        raise ValueError(\'Expected 50-node neighborhoods\')\n    if not np.isfinite(px).all() or not np.isfinite(py).all():\n        raise ValueError(\'Nonfinite coordinates\')\n    dist = np.hypot(px[:, 1:] - px[:, :1], py[:, 1:] - py[:, :1])\n    radius = dist.max(axis=1)\n    normalized = np.divide(dist, radius[:, None], out=np.zeros_like(dist), where=radius[:, None] > 0)\n    bins = np.minimum(np.floor(normalized / BIN_WIDTH).astype(np.int64), N_BINS - 1)\n    counts = np.zeros((len(types), N_BINS, 2), dtype=np.int64)\n    for b in range(N_BINS):\n        for k, ident in enumerate(sender_ids):\n            counts[:, b, k] = ((bins == b) & (types[:, 1:] == ident)).sum(axis=1)\n    per_bin = counts.min(axis=2)\n    m = per_bin.sum(axis=1)\n    original_m = counts.sum(axis=1).min(axis=1)\n    assert np.all(m <= original_m)\n    return dict(distance=dist, normalized=normalized, radius=radius, bins=bins,\n                per_bin=per_bin, m=m, original_m=original_m)\n\n\ndef draw_masks(types, bins, per_bin, dataset_ids, sender_ids, repeat, policy):\n    """Return remove masks (condition, receiver, node); shared by both operators."""\n    if policy not in POLICIES:\n        raise ValueError(policy)\n    masks = np.zeros((3, len(types), 50), dtype=bool)\n    for r, dataset_id in enumerate(dataset_ids):\n        m = int(per_bin[r].sum())\n        if m < 1:\n            raise ValueError(\'Receiver has no overlapping distance bins\')\n        for c in range(3):\n            rng = np.random.default_rng(np.random.SeedSequence([MASK_SEED, int(dataset_id), int(repeat), POLICIES.index(policy), c]))\n            allowed = np.ones(49, dtype=bool) if c == 2 else types[r, 1:] == sender_ids[c]\n            if policy == POLICIES[0]:\n                chosen = rng.choice(np.flatnonzero(allowed), size=m, replace=False)\n            else:\n                chosen = np.concatenate([\n                    rng.choice(np.flatnonzero(allowed & (bins[r] == b)), size=int(k), replace=False)\n                    for b, k in enumerate(per_bin[r]) if k\n                ])\n            masks[c, r, chosen + 1] = True\n            if len(chosen) != m or len(np.unique(chosen)) != m:\n                raise AssertionError(\'Wrong removal count\')\n            if policy == POLICIES[1]:\n                if not np.array_equal(np.bincount(bins[r, chosen], minlength=N_BINS), per_bin[r]):\n                    raise AssertionError(\'Distance-bin counts differ\')\n    if masks[:, :, 0].any() or not np.all(masks.sum(axis=2) == per_bin.sum(axis=1)[None, :]):\n        raise AssertionError(\'Receiver selected or unequal removal counts\')\n    return masks\n', 'r15_replay.py': '"""Exact single-layer free-decoder replay, checked against full forwards at runtime."""\nimport torch\nimport torch.nn.functional as F\n\n\nclass MessageReplay:\n    def __init__(self, model):\n        if len(model.encoders) != 1 or not model.program_aware or model.message_decoder != \'free\' or model.routing_mode != \'program\':\n            raise ValueError(\'Replay is restricted to the verified one-layer program-aware free-decoder checkpoint\')\n        self.model = model\n        self.encoder = model.encoders[0]\n        self.heads = list(self.encoder.attentions.attentions)\n        self.handles = []\n        self.cache = None\n        for h, head in enumerate(self.heads):\n            self.handles.append(head.W_V.register_forward_hook(self._value_hook(h)))\n            self.handles.append(head.W_Eo.register_forward_pre_hook(self._edge_hook(h)))\n            self.handles.append(head.W_program_edge.register_forward_hook(self._projection_hook(h, \'edge_projection\')))\n            self.handles.append(head.W_program_receiver.register_forward_hook(self._projection_hook(h, \'receiver_projection\')))\n            self.handles.append(head.register_forward_hook(self._state_hook(h)))\n\n    def _value_hook(self, h):\n        def hook(module, inputs, output):\n            if self.cache is not None:\n                self.cache[h][\'V\'] = output.detach()\n        return hook\n\n    def _edge_hook(self, h):\n        def hook(module, inputs):\n            if self.cache is not None:\n                self.cache[h][\'edge\'] = inputs[0][:, 0].detach().clone()\n        return hook\n\n    def _projection_hook(self, h, key):\n        def hook(module, inputs, output):\n            if self.cache is not None:\n                self.cache[h][key] = output[:, 0].detach().clone()\n        return hook\n\n    def _state_hook(self, h):\n        def hook(module, inputs, output):\n            if self.cache is not None:\n                self.cache[h][\'alpha\'] = output[2][\'attention\'][:, 0].detach().clone()\n                values = self.cache[h]\n                values[\'logits\'] = torch.einsum(\'bje,be,ke->bjk\', values[\'edge_projection\'],\n                    values[\'receiver_projection\'], inputs[1]) / (module.base_edge_dim ** 0.5)\n                del values[\'edge_projection\'], values[\'receiver_projection\']\n        return hook\n\n    @torch.inference_mode()\n    def capture(self, batch):\n        self.cache = [{} for _ in self.heads]\n        prediction, info = self.model(batch)\n        cache = self.cache\n        self.cache = None\n        g = info[\'activations\'].detach()\n        return dict(heads=cache, gates=g / (g.sum(dim=-1, keepdim=True) + 1e-8),\n                    offset=info[\'baseline\'].detach() + info[\'program_output\'].detach(),\n                    prediction=prediction.detach(), reported_attention=info[\'program_attention\'].detach(),\n                    reported_activations=g, reported_baseline=info[\'baseline\'].detach())\n\n    @torch.inference_mode()\n    def predict(self, cache, remove, operator):\n        if operator not in (\'edge_exclusion\', \'fixed_weight_messages\'):\n            raise ValueError(operator)\n        keep = (~remove).unsqueeze(-1)\n        messages = []\n        for head, values in zip(self.heads, cache[\'heads\']):\n            if operator == \'edge_exclusion\':\n                # Recompute softmax from cached scores, avoiding division by a tiny retained mass.\n                alpha = F.softmax(values[\'logits\'].masked_fill(~keep, float(\'-inf\')), dim=1)\n            else:\n                alpha = values[\'alpha\'] * keep\n            node_message = torch.einsum(\'bjk,bjd->bkd\', alpha, values[\'V\'])\n            edge_message = torch.einsum(\'bjk,bje->bke\', alpha, values[\'edge\'])\n            # Shared affine bias is preserved; only sender-dependent evidence is neutralized.\n            messages.append(node_message + head.W_En(edge_message))\n        head_weights = F.softmax(self.encoder.attentions.W_hn.squeeze(-1), dim=0)\n        combined = (torch.stack(messages, dim=-1) * head_weights).sum(dim=-1)\n        fused = (combined * cache[\'gates\'].unsqueeze(-1)).sum(dim=1)\n        result = cache[\'offset\'] + self.model.program_head.message_to_genes(fused)\n        if not torch.isfinite(result).all():\n            raise ValueError(\'Nonfinite replay prediction\')\n        return result\n\n    def close(self):\n        for h in self.handles:\n            h.remove()\n        self.handles = []\n\n\ndef patched_neutral_forward(attention_source, attention_module):\n    """Build a separately executed full-model control from the audited original method."""\n    import ast\n    import textwrap\n    tree = ast.parse(attention_source)\n    cls = next(n for n in tree.body if isinstance(n, ast.ClassDef) and n.name == \'GRIT_attention\')\n    fn = next(n for n in cls.body if isinstance(n, ast.FunctionDef) and n.name == \'forward\')\n    source = textwrap.dedent(\'\\n\'.join(attention_source.splitlines()[fn.lineno - 1:fn.end_lineno]))\n    marker = \'    # Aggregate node and edge evidence independently for each program:\'\n    if source.count(marker) != 1:\n        raise ValueError(\'Frozen attention implementation differs from the audited version\')\n    insertion = \'\'\'    message_attention = program_attention\n    neutral_mask = getattr(self, \'_r15_remove\', None)\n    if neutral_mask is not None:\n        if program_edge_mask is not None:\n            raise ValueError(\'Do not combine interventions\')\n        message_attention = program_attention.clone()\n        message_attention[:, 0] = message_attention[:, 0] * (~neutral_mask).unsqueeze(-1)\n\n\'\'\'\n    source = source.replace(marker, insertion + marker)\n    source = source.replace(\'"bijk,bjd->bikd", program_attention, V\', \'"bijk,bjd->bikd", message_attention, V\')\n    source = source.replace(\'"bijk,bije->bike", program_attention, edge\', \'"bijk,bije->bike", message_attention, edge\')\n    if source.count(\'message_attention, V\') != 1 or source.count(\'message_attention, edge\') != 1:\n        raise ValueError(\'Expected aggregation sites not found\')\n    namespace = dict(vars(attention_module))\n    exec(compile(source, \'<audited-neutral-message-forward>\', \'exec\'), namespace)\n    return namespace[\'forward\']\n\n\n@torch.inference_mode()\ndef validate_replay(model, replay, batch, selection_masks, attention_source, attention_module):\n    """Full original and separately patched forwards must agree before analyses run."""\n    import types\n    cache = replay.capture(batch)\n    full = cache[\'prediction\']\n    reports = []\n\n    def check(name, actual, expected):\n        error = float((actual - expected).abs().max())\n        ok = bool(torch.allclose(actual, expected, atol=3e-6, rtol=2e-5))\n        reports.append(dict(check=name, maximum_absolute_difference=error, passed=ok))\n        if not ok:\n            raise ValueError(f\'Forward-equivalence check failed: {name}; max error {error}\')\n\n    empty = torch.zeros(full.shape[0], 50, dtype=torch.bool, device=full.device)\n    check(\'identity_edge_replay\', replay.predict(cache, empty, \'edge_exclusion\'), full)\n    check(\'identity_message_replay\', replay.predict(cache, empty, \'fixed_weight_messages\'), full)\n    full_mask = torch.ones(full.shape[0], 50, 50, model.n_programs, dtype=torch.bool, device=full.device)\n    check(\'original_all_edges_present\', model(batch, program_edge_mask=full_mask)[0], full)\n\n    # Contains actual count-only and distance-bin masks, plus one fixed neighbor.\n    cases = list(selection_masks)\n    one = empty.clone(); one[:, 1] = True\n    cases.append((\'one_neighbor\', one))\n    for name, remove in cases:\n        allowed = full_mask.clone()\n        allowed[:, 0] = (~remove).unsqueeze(-1).expand(-1, -1, model.n_programs)\n        reference = model(batch, program_edge_mask=allowed)[0]\n        check(\'edge_\' + name, replay.predict(cache, remove, \'edge_exclusion\'), reference)\n\n    # Patch only in-memory copies of methods, restore even on an exception.\n    forward = patched_neutral_forward(attention_source, attention_module)\n    originals = [head.forward for head in replay.heads]\n    try:\n        for head in replay.heads:\n            head.forward = types.MethodType(forward, head)\n        check(\'neutral_full_identity\', model(batch)[0], full)\n        for name, remove in cases:\n            for head in replay.heads:\n                head._r15_remove = remove\n            reference, info = model(batch)\n            check(\'message_\' + name, replay.predict(cache, remove, \'fixed_weight_messages\'), reference)\n            for key, field in [(\'program_attention\',\'reported_attention\'), (\'activations\',\'reported_activations\'), (\'baseline\',\'reported_baseline\')]:\n                unchanged = bool(torch.equal(info[key], cache[field]))\n                reports.append(dict(check=\'neutral_preserves_\' + key + \'_\' + name, passed=unchanged))\n                if not unchanged:\n                    raise ValueError(\'Fixed-weight control changed \' + key)\n    finally:\n        for head, original in zip(replay.heads, originals):\n            head.forward = original\n            if hasattr(head, \'_r15_remove\'):\n                delattr(head, \'_r15_remove\')\n    check(\'original_forward_restored\', model(batch)[0], full)\n    return reports\n\n\ndef synthetic_replay_test(model_class, attention_source, attention_module):\n    """Runs in Colab before reading the large data file; no optimizer or training."""\n    torch.manual_seed(20260928)\n    genes = [\'gene_a\', \'gene_b\', \'gene_c\', \'gene_d\']\n    ligands = ([[\'gene_a\'], [\'gene_b\']], [[0], [0]])\n    model = model_class(genes, ligands, node_dim=8, edge_dim=4, num_heads=2, n_layers=1,\n        att_dim=2, n_programs=3, program_aware=True, program_token_dim=5,\n        message_decoder=\'free\', routing_mode=\'program\').cuda().eval()\n    batch = dict(x=torch.randn(4,50,4,device=\'cuda\') * .1,\n        type_exp=torch.rand(4,50,4,device=\'cuda\') + 1,\n        cell_types=torch.randint(0,3,(4,50),device=\'cuda\'),\n        position_x=torch.rand(4,50,device=\'cuda\') * 20,\n        position_y=torch.rand(4,50,device=\'cuda\') * 20,\n        y=torch.zeros(4,4,device=\'cuda\'))\n    remove = torch.zeros(4,50,dtype=torch.bool,device=\'cuda\'); remove[:, [2,4,7,11]] = True\n    all_neighbors = torch.ones_like(remove); all_neighbors[:,0] = False\n    replay = MessageReplay(model)\n    try:\n        reports = validate_replay(model, replay, batch, [(\'four_neighbors\',remove),(\'all_noncenter_neighbors\',all_neighbors)], attention_source, attention_module)\n    finally:\n        replay.close()\n    del model, batch, replay\n    torch.cuda.empty_cache()\n    torch.manual_seed(123)\n    return reports\n', 'r15_worker.py': '"""R1.5 fixed-checkpoint controls; all outputs go to a new timestamped directory."""\nfrom pathlib import Path\nfrom datetime import datetime, timezone\nimport argparse\nimport gc\nimport hashlib\nimport json\nimport shutil\nimport time\nimport traceback\nimport uuid\nimport zipfile\nimport numpy as np\nimport pandas as pd\nimport torch\nfrom torch.utils.data import Dataset, DataLoader\nfrom r15_core import *\nfrom r15_replay import MessageReplay, validate_replay, synthetic_replay_test\nfrom spagat_r15.gene_program_model import SpaGP\nimport spagat_r15.attention as attention_module\n\nBASE = Path(\'/content/drive/MyDrive\')\nDATA = BASE / \'GITIII-main/liver_workdir/gitiii/data/processed\'\nPRIOR = BASE / \'SpaGAT_revision/R3_5_7_gene_signed_20260928_090940_UTC\'\nCHECKPOINT = BASE / \'spagatv2/results/liver_free_k4_3epochs/free_seed123_best.pth\'\nCHECKPOINT_SHA = \'2ab3dc98a627ca89c5b17179f71ce57ea47a5f2e36edd84eaab93434191228fe\'\nSPLIT_SHA = \'ac43227801123d1e2bceaa51ba230ec800062767124346c0d51753cf43eefaac\'\nBATCH_SIZE = 32\nREPRO_ATOL = 1e-5\nREPRO_RTOL = 2e-5\nREPRO_MSE_ATOL = 1e-6\nEXPECTED_COUNTS = {\'tumor_1\': 19659, \'tumor_2\': 6892}\nEXPECTED_FULL_MSE = {\'tumor_1\': 0.15171709428216743, \'tumor_2\': 0.15173474350052685}\n\n\ndef sha(path):\n    h = hashlib.sha256()\n    with open(path, \'rb\') as f:\n        for block in iter(lambda: f.read(8 * 1024 * 1024), b\'\'):\n            h.update(block)\n    return h.hexdigest()\n\n\ndef write_json(path, value):\n    Path(path).write_text(json.dumps(value, ensure_ascii=False, indent=2, default=str) + \'\\n\')\n\n\ndef load_torch(path):\n    # No arbitrary pickle execution or unsafe fallback.\n    core = np._core if hasattr(np, \'_core\') else np.core\n    allowed = [argparse.Namespace, np.ndarray, np.dtype,\n               (core.multiarray._reconstruct, \'numpy.core.multiarray._reconstruct\'),\n               (core.multiarray._reconstruct, \'numpy._core.multiarray._reconstruct\'),\n               (core.multiarray.scalar, \'numpy.core.multiarray.scalar\'),\n               (core.multiarray.scalar, \'numpy._core.multiarray.scalar\')]\n    if hasattr(np, \'dtypes\'):\n        for name in (\'Int32DType\', \'Int64DType\', \'Float16DType\', \'Float32DType\', \'Float64DType\', \'StrDType\', \'BoolDType\'):\n            if hasattr(np.dtypes, name):\n                allowed.append(getattr(np.dtypes, name))\n    with torch.serialization.safe_globals(allowed):\n        return torch.load(path, map_location=\'cpu\', weights_only=True)\n\n\nclass CarcinomaData(Dataset):\n    """Same CancerousLiver rows and global type IDs as the already verified export."""\n    def __init__(self, cache, genes, names, meta):\n        self.genes, self.names = list(genes), list(names)\n        self.type_ids = {name: i for i, name in enumerate(names)}\n        n = 460436\n        self.exp = np.lib.format.open_memmap(cache / \'expression.npy\', mode=\'w+\', dtype=\'float32\', shape=(n, len(genes)))\n        self.neighbors = np.lib.format.open_memmap(cache / \'neighbors.npy\', mode=\'w+\', dtype=\'int64\', shape=(n, 50))\n        self.types = np.empty(n, np.int64)\n        self.x, self.y = np.empty(n, np.float32), np.empty(n, np.float32)\n        flags = np.empty(n, bool)\n        path = DATA / \'CancerousLiver.csv\'\n        if path.stat().st_size != 9623364142:\n            raise ValueError(\'CancerousLiver.csv size differs from the audited export\')\n        with np.load(DATA / \'CancerousLiver_TypeExp.npz\', allow_pickle=False) as source:\n            self.baseline = np.zeros((len(names), len(genes)), np.float32)\n            available = set(source.files)\n            for name in available:\n                if name not in self.type_ids or source[name].shape != (len(genes),):\n                    raise ValueError(\'Unexpected cell-type baseline\')\n                self.baseline[self.type_ids[name]] = source[name]\n        if not np.isfinite(self.baseline).all():\n            raise ValueError(\'Nonfinite baselines\')\n        header = list(pd.read_csv(path, nrows=0).columns)\n        required = list(genes) + [\'subclass\', \'centerx\', \'centery\'] + [f\'index_{j}\' for j in range(50)]\n        if not set(required).issubset(header):\n            raise ValueError(\'Processed CSV schema differs from the verified source\')\n        # Use the two parsing passes from the verified R3 export exactly.\n        # Metadata was read separately there, rather than alongside gene columns.\n        metadata_columns = [\'subclass\', \'centerx\', \'centery\'] + ([\'flag\'] if \'flag\' in header else [])\n        print(\'Reading metadata with the original R3 export parser.\', flush=True)\n        metadata = pd.read_csv(path, usecols=metadata_columns)\n        if len(metadata) != n or metadata.subclass.isna().any() or not set(metadata.subclass).issubset(available):\n            raise ValueError(\'Unexpected metadata rows or baseline coverage\')\n        flag = metadata[\'flag\'] if \'flag\' in metadata else pd.Series(True, index=metadata.index)\n        if flag.isna().any() or not flag.isin([True, False, 0, 1]).all():\n            raise ValueError(\'Unexpected receiver eligibility flag\')\n        flags[:] = flag.to_numpy(bool)\n        self.types[:] = metadata.subclass.map(self.type_ids).to_numpy(np.int64)\n        self.x[:], self.y[:] = metadata.centerx.to_numpy(np.float32), metadata.centery.to_numpy(np.float32)\n        columns = list(genes) + [f\'index_{j}\' for j in range(50)]\n        offset = 0\n        print(\'Caching expression and neighbors with the original R3 export parser.\', flush=True)\n        for frame in pd.read_csv(path, usecols=columns, chunksize=4000):\n            end = offset + len(frame)\n            if end > n:\n                raise ValueError(\'Unexpected source row count\')\n            values = frame.loc[:, genes].to_numpy(np.float32)\n            neighbors = frame.loc[:, [f\'index_{j}\' for j in range(50)]].to_numpy()\n            if not np.isfinite(values).all() or not np.isfinite(neighbors).all() or not np.equal(neighbors, np.floor(neighbors)).all():\n                raise ValueError(\'Invalid expression or neighbor indices\')\n            neighbors = neighbors.astype(np.int64)\n            if neighbors.min() < 0 or neighbors.max() >= n or not np.array_equal(neighbors[:, 0], np.arange(offset, end)):\n                raise ValueError(\'Invalid center/neighbor row mapping\')\n            self.exp[offset:end] = values\n            self.neighbors[offset:end] = neighbors\n            offset = end\n            if offset % 80000 == 0:\n                print(f\'  Cached {offset:,}/{n:,} source rows\', flush=True)\n        if offset != n or int(flags.sum()) != 460429:\n            raise ValueError(\'CancerousLiver source/eligible row counts changed\')\n        self.exp.flush(); self.neighbors.flush()\n        eligible = np.flatnonzero(flags)\n        if not np.array_equal(eligible[meta.dataset_index.to_numpy()], meta.csv_row.to_numpy()):\n            raise ValueError(\'Dataset index to CSV row mapping differs from prior export\')\n        if not np.array_equal(np.asarray(names)[self.types[meta.csv_row]], meta.cell_type.to_numpy()):\n            raise ValueError(\'Receiver labels differ from prior export\')\n        self.meta = meta.reset_index(drop=True)\n        self.rows = self.meta.csv_row.to_numpy(np.int64)\n\n    def __len__(self):\n        return len(self.rows)\n\n    def __getitem__(self, i):\n        nb = self.neighbors[self.rows[i]]\n        types = self.types[nb]\n        exp = np.asarray(self.exp[nb], dtype=np.float32)\n        return dict(x=exp, type_exp=self.baseline[types], y=exp[0], cell_types=types,\n                    position_x=self.x[nb], position_y=self.y[nb], neighbor_mask=np.ones(50, np.float32))\n\n\ndef check_saved_targets(ds, saved_target, genes, out):\n    """Check every saved target before spending time on control inference."""\n    for start in range(0, len(ds), 1024):\n        end = min(start + 1024, len(ds))\n        actual = np.asarray(ds.exp[ds.rows[start:end]])\n        expected = np.asarray(saved_target[start:end])\n        report, bad = discrepancy_report(actual, expected, atol=0., rtol=0.)\n        if not report[\'passed\']:\n            report.update(stage=\'target_check\', original_export_row_start=start)\n            write_json(out / \'target_discrepancy.json\', report)\n            write_discrepancy_examples(actual, expected, bad, ds.meta.iloc[start:end], genes,\n                                       out / \'target_discrepancy_examples.csv\')\n            raise ValueError(\'Saved targets differ: see target_discrepancy.json and examples; no predictions were accepted\')\n    return dict(passed=True, n_receivers=len(ds), n_genes=len(genes), exact_equality=True)\n\n\ndef write_discrepancy_examples(actual, expected, bad, metadata, genes, path):\n    rows, columns = np.where(bad)\n    examples = []\n    for i, j in zip(rows[:100], columns[:100]):\n        record = metadata.iloc[int(i)]\n        examples.append(dict(dataset_index=int(record.dataset_index), csv_row=int(record.csv_row),\n                             cell_type=record.cell_type, gene=genes[int(j)],\n                             current=float(actual[i,j]), previous=float(expected[i,j]),\n                             absolute_difference=float(abs(float(actual[i,j])-float(expected[i,j])))))\n    pd.DataFrame(examples).to_csv(path, index=False)\n\n\n@torch.inference_mode()\ndef verify_original_export(model, ds, saved_pred, saved_target, genes, out):\n    """Keep all original export rows and original batch boundaries.\n\n    Version 3 uses the empirically audited numerical reproduction tolerance,\n    while separately checking exact targets and per-population aggregate MSE.\n    The stricter original elementwise discrepancies are still reported.\n    """\n    reports = []\n    first_failure_saved = False\n    population_sums = {name: dict(current_sse=0., previous_sse=0., n_receivers=0)\n                       for name in CONDITIONS[:2]}\n    started = time.perf_counter()\n    for bi, batch in enumerate(DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)):\n        start = bi * BATCH_SIZE\n        end = start + len(batch[\'y\'])\n        current = model({k:v.cuda() for k,v in batch.items()})[0].cpu().numpy()\n        expected = np.asarray(saved_pred[start:end])\n        report, bad = discrepancy_report(current, expected, atol=REPRO_ATOL, rtol=REPRO_RTOL)\n        strict_report, _ = discrepancy_report(current, expected, atol=3e-6, rtol=REPRO_RTOL)\n        report[\'original_tolerance_n_failing_values\'] = strict_report[\'n_failing_values\']\n        report[\'original_tolerance_n_failing_rows\'] = strict_report[\'n_failing_rows\']\n        report.update(batch=bi, original_export_row_start=start, n_receivers=end-start)\n        reports.append(report)\n        metadata = ds.meta.iloc[start:end]\n        truth = np.asarray(saved_target[start:end]).astype(np.float64)\n        has_both = (metadata.n_tumor1_neighbors.to_numpy() > 0) & (metadata.n_tumor2_neighbors.to_numpy() > 0)\n        for name, sums in population_sums.items():\n            selected = has_both & (metadata.cell_type.to_numpy() == name)\n            sums[\'n_receivers\'] += int(selected.sum())\n            sums[\'current_sse\'] += float(np.square(current[selected].astype(np.float64) - truth[selected]).sum())\n            sums[\'previous_sse\'] += float(np.square(expected[selected].astype(np.float64) - truth[selected]).sum())\n        if not report[\'passed\'] and not first_failure_saved:\n            metadata = ds.meta.iloc[start:end]\n            write_discrepancy_examples(current, expected, bad, metadata, genes,\n                                       out / \'prediction_discrepancy_examples.csv\')\n            metadata.to_csv(out / \'first_failed_original_batch_metadata.csv\', index=False)\n            np.savez_compressed(out / \'first_failed_original_batch_predictions.npz\',\n                                current=current, previous=expected,\n                                target=np.asarray(saved_target[start:end]),\n                                dataset_indices=metadata.dataset_index.to_numpy(np.int64))\n            first_failure_saved = True\n        if (bi+1) % 250 == 0:\n            print(f\'  Original-order prediction check: {end:,}/{len(ds):,} rows\', flush=True)\n            pd.DataFrame(reports).to_csv(out / \'prediction_reproduction_batches.csv\', index=False)\n    frame = pd.DataFrame(reports)\n    frame.to_csv(out / \'prediction_reproduction_batches.csv\', index=False)\n    populations = {}\n    for name, sums in population_sums.items():\n        n = sums[\'n_receivers\']\n        current_mse = sums[\'current_sse\'] / (n * len(genes)) if n else float(\'nan\')\n        previous_mse = sums[\'previous_sse\'] / (n * len(genes)) if n else float(\'nan\')\n        passed = (n == EXPECTED_COUNTS[name] and\n                  np.isclose(current_mse, previous_mse, atol=REPRO_MSE_ATOL, rtol=0.) and\n                  np.isclose(previous_mse, EXPECTED_FULL_MSE[name], atol=1e-6, rtol=1e-5))\n        populations[name] = dict(n_receivers=n, current_MSE=current_mse,\n                                previous_MSE=previous_mse, absolute_MSE_difference=abs(current_mse-previous_mse),\n                                passed=bool(passed))\n    mse_passed = all(item[\'passed\'] for item in populations.values())\n    result = dict(passed=bool(frame.passed.all()) and mse_passed,\n                  elementwise_passed=bool(frame.passed.all()), population_MSE_passed=mse_passed,\n                  population_MSE_checks=populations, population_MSE_absolute_tolerance=REPRO_MSE_ATOL,\n                  n_receivers=int(frame.n_receivers.sum()),\n                  n_failing_batches=int((~frame.passed).sum()),\n                  n_failing_rows=int(frame.n_failing_rows.sum()),\n                  n_failing_values=int(frame.n_failing_values.sum()),\n                  maximum_absolute_difference=float(frame.maximum_absolute_difference.max()),\n                  mean_absolute_difference=float(np.average(frame.mean_absolute_difference, weights=frame.n_receivers)),\n                  original_tolerance_n_failing_values=int(frame.original_tolerance_n_failing_values.sum()),\n                  original_tolerance_n_failing_rows=int(frame.original_tolerance_n_failing_rows.sum()),\n                  elapsed_seconds=time.perf_counter()-started, atol=REPRO_ATOL, rtol=REPRO_RTOL,\n                  batching=\'All original tumor export rows in their original order, batch size 32\')\n    write_json(out / \'prediction_reproduction.json\', result)\n    return result\n\n\ndef subset_cache(cache, indices):\n    return dict(heads=[{k:v[indices] for k,v in h.items()} for h in cache[\'heads\']],\n                gates=cache[\'gates\'][indices], offset=cache[\'offset\'][indices],\n                prediction=cache[\'prediction\'][indices])\n\n\ndef summarize(acc, audit, out):\n    rows = []\n    values = acc.cpu().numpy()\n    for ri, receiver in enumerate(CONDITIONS[:2]):\n        for pi, policy in enumerate(POLICIES):\n            for oi, operator in enumerate(OPERATORS):\n                for ci, condition in enumerate(CONDITIONS):\n                    for repeat in range(REPEATS):\n                        sse, abs_change, signed_change, full_sse, n = values[ri, pi, oi, ci, repeat]\n                        if n < 1:\n                            raise ValueError(\'No receiver observations in a required analysis cell\')\n                        entries = n * 1000\n                        rows.append(dict(receiver=receiver, selection=policy, operator=operator, condition=condition,\n                            repeat=repeat, n_receivers=int(n), full_MSE=full_sse/entries, masked_MSE=sse/entries,\n                            delta_MSE=(sse-full_sse)/entries, relative_MSE_increase_percent=100*(sse/full_sse-1),\n                            mean_absolute_prediction_change=abs_change/entries, mean_signed_prediction_change=signed_change/entries))\n    repeats = pd.DataFrame(rows)\n    repeats.to_csv(out / \'control_repeats.csv\', index=False)\n    keys = [\'receiver\', \'selection\', \'operator\', \'condition\']\n    repeats.groupby(keys).agg(n_receivers=(\'n_receivers\', \'first\'),\n        mean_delta_MSE=(\'delta_MSE\', \'mean\'), SD_delta_MSE=(\'delta_MSE\', \'std\'),\n        mean_relative_MSE_increase_percent=(\'relative_MSE_increase_percent\', \'mean\'),\n        SD_relative_MSE_increase_percent=(\'relative_MSE_increase_percent\', \'std\'),\n        mean_absolute_prediction_change=(\'mean_absolute_prediction_change\', \'mean\'),\n        SD_absolute_prediction_change=(\'mean_absolute_prediction_change\', \'std\'),\n        mean_signed_prediction_change=(\'mean_signed_prediction_change\', \'mean\')).reset_index().to_csv(out / \'control_summary.csv\', index=False)\n    paired_keys = [\'receiver\', \'selection\', \'operator\', \'repeat\']\n    contrasts = []\n    for field in [\'relative_MSE_increase_percent\', \'mean_absolute_prediction_change\']:\n        wide = repeats.pivot(index=paired_keys, columns=\'condition\', values=field)\n        for label, a, b in [(\'tumor2_minus_tumor1\', \'tumor_2\', \'tumor_1\'),\n                            (\'tumor1_minus_random\', \'tumor_1\', \'random\'), (\'tumor2_minus_random\', \'tumor_2\', \'random\')]:\n            item = (wide[a] - wide[b]).rename(\'difference\').reset_index()\n            item[\'metric\'], item[\'contrast\'] = field, label\n            contrasts.append(item)\n    contrasts = pd.concat(contrasts, ignore_index=True)\n    contrasts.to_csv(out / \'paired_contrast_repeats.csv\', index=False)\n    contrasts.groupby([\'receiver\', \'selection\', \'operator\', \'metric\', \'contrast\']).difference.agg([\'mean\', \'std\', \'min\', \'max\']).reset_index().to_csv(out / \'paired_contrast_summary.csv\', index=False)\n    balance = []\n    for (receiver, policy, condition, repeat), v in audit.items():\n        nd, nr, rms, active_rms, count, min_d, max_d = v\n        balance.append(dict(receiver=receiver, selection=policy, condition=condition, repeat=repeat,\n            n_selected_sender_instances=int(count), mean_distance_raw=nd/count,\n            mean_distance_over_neighborhood_radius=nr/count, mean_model_input_expression_RMS=rms/count,\n            mean_model_visible_residual_RMS=active_rms/count, min_distance_raw=min_d, max_distance_raw=max_d))\n    pd.DataFrame(balance).to_csv(out / \'selected_sender_balance.csv\', index=False)\n    return repeats\n\n\ndef plot_repeats(repeats, prior, out):\n    import matplotlib\n    matplotlib.use(\'Agg\')\n    import matplotlib.pyplot as plt\n    colors = [\'#20708A\', \'#C96E31\', \'#777777\']\n    fig, axes = plt.subplots(2, 2, figsize=(10, 7), sharex=True)\n    for ri, receiver in enumerate(CONDITIONS[:2]):\n        for oi, operator in enumerate(OPERATORS):\n            ax = axes[ri, oi]\n            frame = repeats[(repeats.receiver == receiver) & (repeats.operator == operator)]\n            for ci, condition in enumerate(CONDITIONS):\n                for pi, policy in enumerate(POLICIES):\n                    y = frame[(frame.condition == condition) & (frame.selection == policy)].sort_values(\'repeat\').relative_MSE_increase_percent.to_numpy()\n                    center = pi * 4 + ci\n                    ax.scatter(center + np.linspace(-.13, .13, len(y)), y, s=14, color=colors[ci], alpha=.7)\n                    ax.errorbar(center, y.mean(), yerr=y.std(ddof=1), color=\'black\', fmt=\'_\', capsize=3)\n            ax.axhline(0, lw=.7, color=\'#999999\')\n            ax.set_title(receiver + \' | \' + operator.replace(\'_\', \' \'), fontsize=10)\n            ax.set_xticks([0,1,2,4,5,6], [\'T1\',\'T2\',\'Random\',\'T1\',\'T2\',\'Random\'])\n            ax.set_xlabel(\'Count only (same m)        Count + distance bins\')\n            ax.set_ylabel(\'MSE increase relative to full prediction (%)\')\n    fig.suptitle(\'Same matched receiver cohort and removal counts; dots = 10 masking repeats\', fontsize=11)\n    fig.tight_layout(); fig.savefig(out / \'new_control_repeat_distributions.png\', dpi=180); plt.close(fig)\n    fig, axes = plt.subplots(2, 2, figsize=(9, 6.5))\n    for ri, receiver in enumerate(CONDITIONS[:2]):\n        frame = prior[prior.receiver == receiver]\n        for mi, (metric, label) in enumerate([(\'relative_MSE_increase_percent\',\'MSE increase (%)\'),\n                                            (\'mean_abs_prediction_change\',\'Mean absolute prediction change\')]):\n            ax = axes[ri, mi]\n            for ci, condition in enumerate(CONDITIONS):\n                y = frame[frame.condition == condition].sort_values(\'repeat\')[metric].to_numpy()\n                if len(y) != 10:\n                    raise ValueError(\'Prior repeat table does not contain all ten repeats\')\n                ax.scatter(ci + np.linspace(-.12, .12, len(y)), y, color=colors[ci], s=20)\n                ax.errorbar(ci, y.mean(), yerr=y.std(ddof=1), color=\'black\', fmt=\'_\', capsize=3)\n            ax.set_title(receiver); ax.set_xticks([0,1,2], [\'Tumor 1\',\'Tumor 2\',\'Random\'])\n            ax.set_ylabel(label)\n    fig.suptitle(\'Original matched-count records: all 10 repeats, mean and sample SD\', fontsize=11)\n    fig.tight_layout(); fig.savefig(out / \'existing_matched_count_repeat_distributions.png\', dpi=180); plt.close(fig)\n\n\ndef run_controls(source_hashes, frozen_attention, historical_repeats_text):\n    stamp = datetime.now(timezone.utc).strftime(\'%Y%m%d_%H%M%S_UTC\') + \'_\' + uuid.uuid4().hex[:6]\n    out = BASE / \'SpaGAT_revision\' / (\'R1_5_distance_message_controls_\' + stamp)\n    out.mkdir(parents=True, exist_ok=False)\n    cache_dir = Path(\'/content\') / (\'R1_5_cache_\' + stamp)\n    cache_dir.mkdir(exist_ok=False)\n    manifest = dict(status=\'running\', analysis_version=\'3_audited_numerical_tolerance\',\n        new_training=False, new_inference=True, original_files_modified=False,\n        checkpoint=str(CHECKPOINT), checkpoint_sha256=CHECKPOINT_SHA, split_sha256=SPLIT_SHA,\n        prior_export=str(PRIOR), source_hashes=source_hashes,\n        torch_version=torch.__version__, numpy_version=np.__version__, batch_size=BATCH_SIZE,\n        repeats=REPEATS, training_seed=123, mask_seed=MASK_SEED,\n        historical_prediction_tolerance=dict(atol=REPRO_ATOL, rtol=REPRO_RTOL,\n                                             population_MSE_atol=REPRO_MSE_ATOL),\n        numerical_tolerance_rationale=\'V2 verified all 76,753,000 targets exactly. Only 5 predictions in 2 rows exceeded atol=3e-6/rtol=2e-5; global max abs difference=7.718801498413086e-6, mean abs difference=4.369752660492709e-8. V3 uses atol=1e-5, retains original discrepancy counts, and adds per-population aggregate MSE checks before controls. No masking outcomes were available when this change was made.\',\n        distance_bin_width_fraction_of_radius=BIN_WIDTH, distance_bins=N_BINS,\n        uncertainty=\'Repeat variation from neighbor selection; not biological confidence intervals.\',\n        scope=\'Within-specimen, existing-checkpoint sensitivity controls. Sender labels and model inputs are unchanged.\',\n        fixed_weight_control=\'Keep original softmax and gates; neutralize chosen sender-dependent node/edge evidence. Shared affine biases remain unchanged.\',\n        matching_scope=\'Distance bins and absolute counts; expression magnitude and removal fractions are audited but not matched.\')\n    started = time.perf_counter()\n    replay = None\n    try:\n        if not torch.cuda.is_available():\n            raise RuntimeError(\'请选择 Colab GPU（例如 T4），本程序不在 CPU 上启动完整推理。\')\n        if BATCH_SIZE != 32:\n            raise ValueError(\'Reproduction requires the original R3 export batch size of 32; do not change it\')\n        if shutil.disk_usage(\'/content\').free < 6 * 1024**3:\n            raise RuntimeError(\'Need at least 6 GiB free local runtime disk space\')\n        manifest[\'gpu\'] = torch.cuda.get_device_name(0)\n        torch.manual_seed(123)\n        torch.backends.cuda.matmul.allow_tf32 = False\n        torch.backends.cudnn.allow_tf32 = False\n        torch.backends.cudnn.benchmark = False\n        manifest[\'synthetic_forward_checks\'] = synthetic_replay_test(SpaGP, frozen_attention, attention_module)\n        print(\'Synthetic model replay/control checks passed.\', flush=True)\n        if sha(CHECKPOINT) != CHECKPOINT_SHA or sha(CHECKPOINT.parent / \'shared_split.json\') != SPLIT_SHA:\n            raise ValueError(\'Checkpoint/split hash differs; no automatic substitution\')\n        cp = load_torch(CHECKPOINT)\n        split = json.loads((CHECKPOINT.parent / \'shared_split.json\').read_text())\n        val = np.asarray(split[\'val_indices\'], np.int64)\n        if not np.array_equal(val, np.asarray(cp[\'shared_split\'][\'val_indices\'])) or len(val) != 158652:\n            raise ValueError(\'Validation split differs from the verified export\')\n        genes = list(load_torch(DATA.parent / \'genes.pth\'))\n        ligands = load_torch(DATA.parent / \'ligands.pth\')\n        prior_provenance = json.loads((PRIOR / \'provenance.json\').read_text())\n        if prior_provenance[\'selected_checkpoint\'] != str(CHECKPOINT):\n            raise ValueError(\'Prior export names a different checkpoint\')\n        if prior_provenance.get(\'batch_size\') != BATCH_SIZE:\n            raise ValueError(\'Prior export batch size differs\')\n        if genes != json.loads((PRIOR / \'genes.json\').read_text()) or len(genes) != 1000:\n            raise ValueError(\'Gene identities/order differ from prior export\')\n        meta_all = pd.read_csv(PRIOR / \'receiver_metadata.csv\')\n        old_ids = np.load(PRIOR / \'arrays/dataset_indices.npy\', allow_pickle=False)\n        if len(meta_all) != 76753 or not np.array_equal(old_ids, meta_all.dataset_index) or not np.array_equal(val[np.isin(val, old_ids)], old_ids):\n            raise ValueError(\'Prior receiver indices/order changed\')\n        if meta_all.dataset_index.duplicated().any() or not (meta_all[\'sample\'] == \'CancerousLiver\').all():\n            raise ValueError(\'Unexpected receiver metadata\')\n        eligible = (meta_all.n_tumor1_neighbors > 0) & (meta_all.n_tumor2_neighbors > 0)\n        positions = np.flatnonzero(eligible)\n        meta = meta_all.iloc[positions].copy().reset_index(drop=True)\n        if meta.cell_type.value_counts().to_dict() != EXPECTED_COUNTS:\n            raise ValueError(\'Eligible receiver counts differ from historical masking\')\n        manifest[\'prior_metadata_sha256\'] = sha(PRIOR / \'receiver_metadata.csv\')\n        for name in [\'prediction.npy\', \'target.npy\']:\n            print(\'Copying existing \' + name + \' to runtime cache.\', flush=True)\n            shutil.copyfile(PRIOR / \'arrays\' / name, cache_dir / name)\n        saved_pred = np.load(cache_dir / \'prediction.npy\', mmap_mode=\'r\', allow_pickle=False)\n        saved_target = np.load(cache_dir / \'target.npy\', mmap_mode=\'r\', allow_pickle=False)\n        if saved_pred.shape != (76753, 1000) or saved_target.shape != saved_pred.shape:\n            raise ValueError(\'Prior arrays have unexpected shapes\')\n        ds = CarcinomaData(cache_dir, genes, prior_provenance[\'cell_type_names\'], meta_all)\n        manifest[\'target_reproduction\'] = check_saved_targets(ds, saved_target, genes, out)\n        print(\'All 76,753 saved target rows match exactly.\', flush=True)\n        write_json(out / \'manifest.json\', manifest)\n        # Keep ds in its original 76,753-row order for identical inference batches.\n        # Matching and metric denominators still use only the original 26,551 eligible receivers.\n        neighbor_ids = ds.neighbors[meta.csv_row.to_numpy(np.int64)]\n        types = ds.types[neighbor_ids]\n        sender_ids = [ds.type_ids[x] for x in CONDITIONS[:2]]\n        for k, column in enumerate([\'n_tumor1_neighbors\', \'n_tumor2_neighbors\']):\n            if not np.array_equal((types[:, 1:] == sender_ids[k]).sum(axis=1), meta[column]):\n                raise ValueError(\'Neighbor composition differs from prior export\')\n        plan = distance_plan(types, ds.x[neighbor_ids], ds.y[neighbor_ids], sender_ids)\n        active = plan[\'m\'] > 0\n        matching = meta.copy()\n        matching[\'original_matched_count\'] = plan[\'original_m\']\n        matching[\'distance_matched_count\'] = plan[\'m\']\n        matching[\'distance_eligible\'] = active\n        matching[\'radius_in_saved_coordinate_units\'] = plan[\'radius\']\n        matching[\'tumor1_removal_fraction\'] = plan[\'m\'] / meta.n_tumor1_neighbors\n        matching[\'tumor2_removal_fraction\'] = plan[\'m\'] / meta.n_tumor2_neighbors\n        matching.to_csv(out / \'receiver_matching.csv\', index=False)\n        coverage = []\n        for receiver in CONDITIONS[:2]:\n            in_type = meta.cell_type.to_numpy() == receiver\n            use = in_type & active\n            coverage.append(dict(receiver=receiver, original_eligible_receivers=int(in_type.sum()),\n                distance_eligible_receivers=int(use.sum()), coverage_fraction=float(use.sum()/in_type.sum()),\n                mean_original_matched_count=float(plan[\'original_m\'][in_type].mean()),\n                mean_control_removal_count=float(plan[\'m\'][use].mean()) if use.any() else None,\n                mean_tumor1_removal_fraction=float(matching.loc[use, \'tumor1_removal_fraction\'].mean()),\n                mean_tumor2_removal_fraction=float(matching.loc[use, \'tumor2_removal_fraction\'].mean())))\n        write_json(out / \'matching_coverage.json\', coverage)\n        manifest[\'matching_coverage\'] = coverage\n        if any(item[\'distance_eligible_receivers\'] == 0 for item in coverage):\n            raise ValueError(\'No overlapping distance support for one receiver population; do not widen bins after inspecting outcomes\')\n        print(\'Matching coverage:\', coverage, flush=True)\n        model = SpaGP(genes=genes, ligands_info=ligands, node_dim=256, edge_dim=48, num_heads=2,\n            n_layers=1, att_dim=8, n_programs=4, program_aware=True, program_token_dim=32,\n            message_decoder=\'free\', routing_mode=\'program\').cuda().eval()\n        model.load_state_dict(cp[\'model\'], strict=True)\n        del cp; gc.collect()\n        print(\'Verifying every original unmasked prediction BEFORE control analysis.\', flush=True)\n        manifest[\'prediction_reproduction\'] = verify_original_export(model, ds, saved_pred, saved_target, genes, out)\n        write_json(out / \'manifest.json\', manifest)\n        if not manifest[\'prediction_reproduction\'][\'passed\']:\n            raise ValueError(\'Original-order prediction check failed; detailed numerical diagnostics are included, controls not started\')\n        print(\'All original predictions and per-population MSEs passed reproduction checks.\', flush=True)\n        replay = MessageReplay(model)\n        chosen = np.concatenate([np.flatnonzero(active & (meta.cell_type.to_numpy() == r))[:8] for r in CONDITIONS[:2]])\n        small = next(iter(DataLoader(torch.utils.data.Subset(ds, positions[chosen].tolist()), batch_size=len(chosen), shuffle=False)))\n        cases = []\n        for policy in POLICIES:\n            masks = draw_masks(types[chosen], plan[\'bins\'][chosen], plan[\'per_bin\'][chosen], meta.dataset_index.to_numpy()[chosen], sender_ids, 0, policy)\n            for ci, condition in enumerate(CONDITIONS):\n                cases.append((policy + \'_\' + condition, torch.as_tensor(masks[ci], device=\'cuda\')))\n        manifest[\'forward_equivalence\'] = validate_replay(model, replay, {k:v.cuda() for k,v in small.items()}, cases, frozen_attention, attention_module)\n        print(\'Full-forward equivalence checks passed for both operators.\', flush=True)\n        write_json(out / \'manifest.json\', manifest)\n        acc = torch.zeros((2, 2, 2, 3, REPEATS, 5), dtype=torch.float64, device=\'cuda\')\n        audit = {}\n        offset = 0; max_pred_error = 0.; max_target_error = 0.\n        full_sse = np.zeros(2); full_counts = np.zeros(2, dtype=np.int64)\n        evaluate_started = time.perf_counter()\n        with torch.inference_mode():\n            for bi, batch in enumerate(DataLoader(ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=0)):\n                sl, local_positions = original_batch_selection(len(ds), positions, bi * BATCH_SIZE, BATCH_SIZE)\n                size = len(sl)\n                if not size:\n                    continue\n                reference = np.asarray(saved_pred[positions[sl]])\n                target_reference = np.asarray(saved_target[positions[sl]])\n                batch_gpu = {k:v.cuda() for k,v in batch.items()}\n                original_cache = replay.capture(batch_gpu)\n                local_gpu = torch.as_tensor(local_positions, device=\'cuda\')\n                cache = subset_cache(original_cache, local_gpu)\n                batch_gpu = {k:v[local_gpu] for k,v in batch_gpu.items()}\n                batch = {k:v[local_positions] for k,v in batch.items()}\n                prediction = cache[\'prediction\']\n                target = batch_gpu[\'y\']\n                original = torch.as_tensor(reference, device=\'cuda\')\n                original_target = torch.as_tensor(target_reference, device=\'cuda\')\n                if not torch.allclose(prediction, original, atol=REPRO_ATOL, rtol=REPRO_RTOL) or not torch.equal(target, original_target):\n                    np.savez_compressed(out / \'control_stage_reproduction_failure.npz\',\n                        current=prediction.cpu().numpy(), previous=reference,\n                        current_target=target.cpu().numpy(), previous_target=target_reference,\n                        original_export_positions=positions[sl])\n                    raise ValueError(\'Control-stage reproduction failed after the full preflight; diagnostic arrays saved\')\n                max_pred_error = max(max_pred_error, float((prediction-original).abs().max()))\n                max_target_error = max(max_target_error, float((target-original_target).abs().max()))\n                receiver_types = meta.cell_type.to_numpy()[sl]\n                for ri, receiver in enumerate(CONDITIONS[:2]):\n                    take = receiver_types == receiver\n                    take_gpu = torch.as_tensor(take, device=\'cuda\')\n                    full_sse[ri] += float(((prediction[take_gpu].double()-target[take_gpu].double())**2).sum())\n                    full_counts[ri] += int(take.sum())\n                use = active[sl]\n                if use.any():\n                    # Subset the captured factors; no resampling of receivers by condition.\n                    use_gpu = torch.as_tensor(use, device=\'cuda\')\n                    small_cache = dict(heads=[{k:v[use_gpu] for k,v in h.items()} for h in cache[\'heads\']],\n                        gates=cache[\'gates\'][use_gpu], offset=cache[\'offset\'][use_gpu], prediction=prediction[use_gpu])\n                    ids = sl[use]\n                    yf = target[use_gpu].double(); pf = prediction[use_gpu].double()\n                    groups = [receiver_types[use] == receiver for receiver in CONDITIONS[:2]]\n                    gpu_groups = [torch.as_tensor(group, device=\'cuda\') for group in groups]\n                    full_errors = (pf-yf).square()\n                    raw_x = batch[\'x\'].numpy()[use]\n                    baseline = batch[\'type_exp\'].numpy()[use]\n                    homo = types[ids] == types[ids, :1]\n                    visible = np.where(homo[..., None], 0., raw_x)\n                    input_rms = np.sqrt(np.mean((baseline.astype(np.float64) + visible)**2, axis=2))\n                    residual_rms = np.sqrt(np.mean(visible.astype(np.float64)**2, axis=2))\n                    for pi, policy in enumerate(POLICIES):\n                        for repeat in range(REPEATS):\n                            masks = draw_masks(types[ids], plan[\'bins\'][ids], plan[\'per_bin\'][ids], meta.dataset_index.to_numpy()[ids], sender_ids, repeat, policy)\n                            if policy == POLICIES[1]:\n                                for row in range(len(ids)):\n                                    d0 = np.sort(plan[\'normalized\'][ids[row]][masks[0,row,1:]])\n                                    for ci in (1, 2):\n                                        d1 = np.sort(plan[\'normalized\'][ids[row]][masks[ci,row,1:]])\n                                        if np.max(np.abs(d0-d1)) > BIN_WIDTH + 1e-12:\n                                            raise AssertionError(\'Distance matching exceeds its declared tolerance\')\n                            for ci, condition in enumerate(CONDITIONS):\n                                remove = torch.as_tensor(masks[ci], device=\'cuda\')\n                                for ri, receiver in enumerate(CONDITIONS[:2]):\n                                    group = groups[ri]\n                                    if not group.any():\n                                        continue\n                                    selected = masks[ci, group, 1:]\n                                    raw_d = plan[\'distance\'][ids[group]][selected]\n                                    norm_d = plan[\'normalized\'][ids[group]][selected]\n                                    input_values = input_rms[group, 1:][selected]\n                                    residual_values = residual_rms[group, 1:][selected]\n                                    key = (receiver, policy, condition, repeat)\n                                    if key not in audit:\n                                        audit[key] = np.array([0.,0.,0.,0.,0.,np.inf,-np.inf])\n                                    audit[key][:5] += [raw_d.sum(), norm_d.sum(), input_values.sum(), residual_values.sum(), len(raw_d)]\n                                    audit[key][5] = min(audit[key][5], raw_d.min())\n                                    audit[key][6] = max(audit[key][6], raw_d.max())\n                                for oi, operator in enumerate(OPERATORS):\n                                    perturbed = replay.predict(small_cache, remove, operator).double()\n                                    change = perturbed - pf\n                                    error = (perturbed-yf).square()\n                                    for ri, group in enumerate(groups):\n                                        n = int(group.sum())\n                                        if n:\n                                            gidx = gpu_groups[ri]\n                                            vals = torch.stack([error[gidx].sum(), change[gidx].abs().sum(), change[gidx].sum(), full_errors[gidx].sum(), error.new_tensor(n)])\n                                            acc[ri, pi, oi, ci, repeat] += vals\n                offset += size\n                if bi == 7 or (bi + 1) % 25 == 0:\n                    torch.cuda.synchronize()\n                    elapsed = time.perf_counter() - evaluate_started\n                    eta = elapsed / offset * (len(meta)-offset) / 60\n                    print(f\'  Evaluated {offset:,}/{len(meta):,} eligible receivers; estimated remaining inference {eta:.1f} min\', flush=True)\n                if (bi + 1) % 100 == 0:\n                    write_json(out / \'progress.json\', dict(receivers_done=offset, receivers_total=len(meta), elapsed_seconds=time.perf_counter()-evaluate_started))\n                    np.save(cache_dir / \'running_metric_sums.npy\', acc.cpu().numpy())\n        verified = {}\n        for ri, receiver in enumerate(CONDITIONS[:2]):\n            value = full_sse[ri] / (full_counts[ri] * 1000)\n            if full_counts[ri] != EXPECTED_COUNTS[receiver] or not np.isclose(value, EXPECTED_FULL_MSE[receiver], atol=1e-6, rtol=1e-5):\n                raise ValueError(\'Full-cohort baseline MSE check failed\')\n            verified[receiver] = dict(n_receivers=int(full_counts[ri]), full_MSE=float(value))\n        manifest[\'baseline_verification\'] = dict(populations=verified, maximum_prediction_difference=max_pred_error, maximum_target_difference=max_target_error)\n        repeats = summarize(acc, audit, out)\n        for item in coverage:\n            if not (repeats.loc[repeats.receiver == item[\'receiver\'], \'n_receivers\'] == item[\'distance_eligible_receivers\']).all():\n                raise AssertionError(\'A condition used a different receiver cohort\')\n        import io\n        old = pd.read_csv(io.StringIO(historical_repeats_text))\n        old.to_csv(out / \'existing_original_matched_count_repeats.csv\', index=False)\n        manifest[\'historical_repeat_table_sha256\'] = hashlib.sha256(historical_repeats_text.encode()).hexdigest()\n        try:\n            plot_repeats(repeats, old, out)\n            manifest[\'plot_status\'] = \'complete\'\n        except Exception as exc:\n            manifest[\'plot_status\'] = \'tables_complete_plotting_failed\'\n            manifest[\'plot_error\'] = str(exc)\n            (out / \'plot_error_trace.txt\').write_text(traceback.format_exc())\n        manifest.update(status=\'complete\', n_control_repeats=len(repeats), elapsed_seconds=time.perf_counter()-started)\n        print(\'Completed. Original model, data, and previous results were not modified.\', flush=True)\n    except Exception as exc:\n        manifest.update(status=\'stopped\', error=str(exc), elapsed_seconds=time.perf_counter()-started)\n        (out / \'error_trace.txt\').write_text(traceback.format_exc())\n        print(\'Stopped:\', str(exc), \'Return the diagnostic ZIP; do not loosen verification thresholds.\', flush=True)\n    finally:\n        if replay is not None:\n            replay.close()\n        source_dir = out / \'source\'\n        source_dir.mkdir(exist_ok=True)\n        code_root = Path(__file__).parent\n        for relative, expected in source_hashes.items():\n            path = code_root / relative\n            if path.is_file():\n                target = source_dir / relative\n                target.parent.mkdir(parents=True, exist_ok=True)\n                shutil.copyfile(path, target)\n        write_json(out / \'manifest.json\', manifest)\n        (out / \'README.txt\').write_text(\n            \'R1.5 sensitivity controls; existing fixed model, no training.\\n\'\n            \'Version 3 retains the original CSV parsing passes and original 76,753-row batch layout.\\n\'\n            \'Targets must be exactly equal. Historical prediction tolerance is atol=1e-5/rtol=2e-5 after auditing the V2 differences.\\n\'\n            \'Original tighter-tolerance discrepancy counts remain reported; population MSE agreement is checked before controls.\\n\'\n            \'Full-forward message equivalence retains its original atol=3e-6/rtol=2e-5; masks, cohorts, model and outcomes are unchanged.\\n\'\n            \'Both new selections use the same distance-eligible receivers and the same removal count per receiver.\\n\'\n            \'Distance bins have width 0.05 of each receiver neighborhood radius; bins are fixed before outcomes.\\n\'\n            \'The random distance control uses the same bin-wise counts. This is not a binned-label-permutation experiment.\\n\'\n            \'Both intervention operators use identical selected sender sets for each repeat.\\n\'\n            \'Fixed-weight messages preserves original softmax/gates and shared biases; it is not a biological cell-removal simulation.\\n\'\n            \'Expression magnitude and unequal removal fractions remain potential differences; balance and coverage are reported.\\n\'\n            \'No physical coordinate units are inferred; raw saved units and distances normalized by neighborhood radius are reported.\\n\'\n            \'Repeat SDs describe neighbor-selection variability, not training or biological replication.\\n\'\n            \'Existing-repeat plots use the original sender-origin repeat records and their original larger eligible cohort; do not treat them as the same cohort as new controls.\\n\'\n            \'Read status and forward/baseline verification in manifest.json before using the results.\\n\')\n        archive = out.with_suffix(\'.zip\')\n        with zipfile.ZipFile(archive, \'x\', zipfile.ZIP_DEFLATED) as z:\n            for path in sorted(out.rglob(\'*\')):\n                if path.is_file():\n                    z.write(path, arcname=str(path.relative_to(out)))\n        print(\'请上传结果 ZIP:\', archive, flush=True)\n    return archive\n'}
HISTORICAL_REPEATS = 'repeat,receiver,condition,n_receivers,mean_edges_removed,mean_abs_prediction_change,full_MSE,masked_MSE,delta_MSE,relative_MSE_increase_percent\n0,tumor_1,tumor_1,19659,6.465588280177018,0.029178346500667054,0.15171709415399504,0.15949054491064532,0.007773450756650285,5.123648590817407\n0,tumor_1,tumor_2,19659,6.465588280177018,0.03766682920274502,0.15171709415399504,0.16194579110741525,0.010228696953420213,6.741954168353592\n0,tumor_1,random,19659,6.465588280177018,0.026618155744163822,0.15171709415399504,0.15812605511161656,0.006408960957621529,4.2242840158251\n0,tumor_2,tumor_1,6892,13.015670342426,0.05294998926670656,0.15173474333525672,0.16845378152091214,0.016719038185655427,11.018595885264618\n0,tumor_2,tumor_2,6892,13.015670342426,0.08659240169869871,0.15173474333525672,0.18950030145462488,0.03776555811936816,24.8891962969387\n0,tumor_2,random,6892,13.015670342426,0.04834743963732442,0.15173474333525672,0.16410576610017225,0.012371022764915529,8.153058747779243\n1,tumor_1,tumor_1,19659,6.465588280177018,0.02932731126968237,0.15171709415399504,0.15918684481046372,0.007469750656468682,4.923473322582081\n1,tumor_1,tumor_2,19659,6.465588280177018,0.03774827386595673,0.15171709415399504,0.16188513605793164,0.010168041903936609,6.701975120625432\n1,tumor_1,random,19659,6.465588280177018,0.025941229740135838,0.15171709415399504,0.15762095463298345,0.005903860478988415,3.8913614262845764\n1,tumor_2,tumor_1,6892,13.015670342426,0.05278014012303331,0.15173474333525672,0.16811031036410162,0.016375567028844906,10.792232990873567\n1,tumor_2,tumor_2,6892,13.015670342426,0.08782651749646342,0.15173474333525672,0.19021911028133046,0.03848436694607374,25.362923546812766\n1,tumor_2,random,6892,13.015670342426,0.05049496640053535,0.15173474333525672,0.16613180583725817,0.014397062502001456,9.488309786896505\n2,tumor_1,tumor_1,19659,6.465588280177018,0.029272245880115358,0.15171709415399504,0.1589478698360622,0.007230775682067175,4.765959776904133\n2,tumor_1,tumor_2,19659,6.465588280177018,0.03768007056386505,0.15171709415399504,0.16194703425746937,0.010229940103474333,6.742773555292851\n2,tumor_1,random,19659,6.465588280177018,0.026272654033612546,0.15171709415399504,0.15744288057625805,0.005725786422263013,3.7739889853487814\n2,tumor_2,tumor_1,6892,13.015670342426,0.05310705385199309,0.15173474333525672,0.1683177262304553,0.016582982895198578,10.928929347814961\n2,tumor_2,tumor_2,6892,13.015670342426,0.08702099127849987,0.15173474333525672,0.19092452710157762,0.0391897837663209,25.82782486390173\n2,tumor_2,random,6892,13.015670342426,0.05009792881128846,0.15173474333525672,0.16596139799960463,0.014226654664347915,9.376003380395375\n3,tumor_1,tumor_1,19659,6.465588280177018,0.029369562032221658,0.15171709415399504,0.15899518103783647,0.007278086883841434,4.797143607597751\n3,tumor_1,tumor_2,19659,6.465588280177018,0.037602784143434584,0.15171709415399504,0.1617850287068902,0.010067934552895175,6.63599221237132\n3,tumor_1,random,19659,6.465588280177018,0.026855125361811054,0.15171709415399504,0.1582977722174178,0.0065806780634227735,4.337466453676796\n3,tumor_2,tumor_1,6892,13.015670342426,0.05329958173566894,0.15173474333525672,0.16871148763502744,0.016976744299770724,11.188435770613683\n3,tumor_2,tumor_2,6892,13.015670342426,0.08640646982459363,0.15173474333525672,0.18906318259529784,0.03732843926004112,24.601115367206457\n3,tumor_2,random,6892,13.015670342426,0.0499957365086967,0.15173474333525672,0.16619685147739882,0.014462108142142105,9.531177780548381\n4,tumor_1,tumor_1,19659,6.465588280177018,0.029828852911144413,0.15171709415399504,0.1590761451633872,0.007359051009392159,4.85050880418433\n4,tumor_1,tumor_2,19659,6.465588280177018,0.03757678394091082,0.15171709415399504,0.16185978313011193,0.010142688976116893,6.685264460589996\n4,tumor_1,random,19659,6.465588280177018,0.027115724374221467,0.15171709415399504,0.1583464085950366,0.006629314441041562,4.369523736272401\n4,tumor_2,tumor_1,6892,13.015670342426,0.05339786064138531,0.15173474333525672,0.16862315761634797,0.016888414281091252,11.130222327378531\n4,tumor_2,tumor_2,6892,13.015670342426,0.08773492552611589,0.15173474333525672,0.19049887498976253,0.03876413165450582,25.54730103497574\n4,tumor_2,random,6892,13.015670342426,0.05129816636058718,0.15173474333525672,0.165951327177237,0.014216583841980296,9.369366256855798\n5,tumor_1,tumor_1,19659,6.465588280177018,0.02974890509517773,0.15171709415399504,0.15945874815793917,0.007741654003944137,5.102690667200789\n5,tumor_1,tumor_2,19659,6.465588280177018,0.037814753242145144,0.15171709415399504,0.1622059029273981,0.010488808773403052,6.913399463581052\n5,tumor_1,random,19659,6.465588280177018,0.026590120624547523,0.15171709415399504,0.15800290240928075,0.006285808255285713,4.1431114208564574\n5,tumor_2,tumor_1,6892,13.015670342426,0.0530222870680356,0.15173474333525672,0.16845210901084365,0.01671736567558693,11.017493626130202\n5,tumor_2,tumor_2,6892,13.015670342426,0.08790035153502945,0.15173474333525672,0.19055530626885916,0.03882056293360245,25.584491778411433\n5,tumor_2,random,6892,13.015670342426,0.049755117416144,0.15173474333525672,0.16691037495276018,0.01517563161750346,10.00142174687904\n6,tumor_1,tumor_1,19659,6.465588280177018,0.028640612380063404,0.15171709415399504,0.158876875998997,0.007159781845001961,4.719166211906669\n6,tumor_1,tumor_2,19659,6.465588280177018,0.037596383776194904,0.15171709415399504,0.16175789891718437,0.010040804763189332,6.618110384448683\n6,tumor_1,random,19659,6.465588280177018,0.025895790603264066,0.15171709415399504,0.15790918612585675,0.006192091971861713,4.08134100273279\n6,tumor_2,tumor_1,6892,13.015670342426,0.053005038680137936,0.15173474333525672,0.16837139203807192,0.0166366487028152,10.9642975215351\n6,tumor_2,tumor_2,6892,13.015670342426,0.08815449017159818,0.15173474333525672,0.1915297480595368,0.039795004724280075,26.226692614725245\n6,tumor_2,random,6892,13.015670342426,0.04832843389346962,0.15173474333525672,0.16580495215652297,0.01407020882126625,9.272898554405721\n7,tumor_1,tumor_1,19659,6.465588280177018,0.02933062718321951,0.15171709415399504,0.15918683340518058,0.007469739251185548,4.9234658051146525\n7,tumor_1,tumor_2,19659,6.465588280177018,0.03774150122479957,0.15171709415399504,0.16205100771560874,0.0103339135616137,6.811304697890291\n7,tumor_1,random,19659,6.465588280177018,0.02655504574130623,0.15171709415399504,0.15836002836281263,0.006642934208817591,4.378500818157589\n7,tumor_2,tumor_1,6892,13.015670342426,0.05279406067800778,0.15173474333525672,0.16820404182141835,0.016469298486161632,10.854006224383856\n7,tumor_2,tumor_2,6892,13.015670342426,0.08662138137088583,0.15173474333525672,0.18969934371924996,0.03796460038399324,25.020374074849\n7,tumor_2,random,6892,13.015670342426,0.05029533730817443,0.15173474333525672,0.16603868850690295,0.014303945171646237,9.426941290592744\n8,tumor_1,tumor_1,19659,6.465588280177018,0.028599041191339068,0.15171709415399504,0.1588956026635824,0.007178508509587372,4.73150935932182\n8,tumor_1,tumor_2,19659,6.465588280177018,0.037551014951944316,0.15171709415399504,0.16191875274586795,0.010201658591872914,6.724132602696756\n8,tumor_1,random,19659,6.465588280177018,0.026872304575053605,0.15171709415399504,0.15774456152160884,0.006027467367613809,3.972833385205653\n8,tumor_2,tumor_1,6892,13.015670342426,0.05293801684980078,0.15173474333525672,0.16816378375533286,0.016429040420076146,10.827474353567338\n8,tumor_2,tumor_2,6892,13.015670342426,0.08685906936511942,0.15173474333525672,0.19064224458363468,0.03890750124837797,25.641788026366612\n8,tumor_2,random,6892,13.015670342426,0.05047426781084256,0.15173474333525672,0.16554273232400799,0.013807988988751269,9.100083926225535\n9,tumor_1,tumor_1,19659,6.465588280177018,0.029369004265832978,0.15171709415399504,0.1588940503998323,0.0071769562458372516,4.730486228896882\n9,tumor_1,tumor_2,19659,6.465588280177018,0.037576258544707454,0.15171709415399504,0.16182999068870021,0.010112896534705179,6.665627621657744\n9,tumor_1,random,19659,6.465588280177018,0.026293552003227454,0.15171709415399504,0.15819453503815215,0.00647744088415711,4.269420608321442\n9,tumor_2,tumor_1,6892,13.015670342426,0.05324565866733542,0.15173474333525672,0.16830365487827648,0.01656891154301976,10.919655695736658\n9,tumor_2,tumor_2,6892,13.015670342426,0.08753495652669228,0.15173474333525672,0.1903320200217718,0.03859727668651508,25.43733612890141\n9,tumor_2,random,6892,13.015670342426,0.04887209523670284,0.15173474333525672,0.16492384000912144,0.013189096673864725,8.692206138131146\n'
CODE_ROOT = Path('/content') / ('spagat_r15_code_' + uuid.uuid4().hex[:8])
CODE_ROOT.mkdir(exist_ok=False)
SOURCE_HASHES = {}
for relative, source in MODULES.items():
    path = CODE_ROOT / relative
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(source)
    SOURCE_HASHES[relative] = hashlib.sha256(source.encode()).hexdigest()
    assert hashlib.sha256(path.read_bytes()).hexdigest() == SOURCE_HASHES[relative]
for name in list(sys.modules):
    if name in {'r15_core','r15_replay','r15_worker'} or name == 'spagat_r15' or name.startswith('spagat_r15.'):
        del sys.modules[name]
sys.path.insert(0, str(CODE_ROOT))
import r15_worker
print('Frozen model and control code ready. Original Drive source is unchanged.')


## 3. 运行并下载

保留下面的默认设置即可。程序将按旧导出流程分别读取约 9.6 GB 癌组织 CSV 的元数据和表达列，并复制先前的预测/target；这些是读取和缓存，不是训练。正常肝大 CSV 不会被读取。

程序先运行小型模型检查，再校验 checkpoint/split 哈希与 receiver 对应关系。全部旧 target 必须逐项相同，旧预测须通过上述数值容差及两类 receiver 的整体 MSE 核对。真实 checkpoint 上的两种消息计算也必须与完整模型一致，然后才执行正式控制分析。

控制阶段保留原始批次，计算一次原始图消息后，仅取合格 receiver 的中间量完成 120 个控制组合；不是把旧汇总值当作新控制结果。核对阶段会显示进度，进入控制分析后会显示基于实测速率的剩余推理时间。

结果写到新的 `MyDrive/SpaGAT_revision/R1_5_distance_message_controls_时间戳/`。完成或核对失败均自动下载 ZIP，**将 ZIP 发回当前对话**。保持 batch=32，与旧导出一致；不要改匹配区间或核对容差强行通过。失败时也会打包诊断信息，无需自行查找文件。


In [ ]:
BATCH_SIZE = 32
r15_worker.BATCH_SIZE = BATCH_SIZE
archive = r15_worker.run_controls(
    SOURCE_HASHES,
    MODULES['spagat_r15/attention.py'],
    HISTORICAL_REPEATS,
)
from google.colab import files
files.download(str(archive))


## 输出与解释

- `manifest.json`：完成/停止状态、checkpoint 与代码身份、完整模型一致性检查、旧预测复现、耗时。只有 `status=complete` 才代表分析完成；绘图失败会另行注明，数值表仍保留。
- `prediction_reproduction.json` / `prediction_reproduction_batches.csv`：全部旧预测的复现结果；如失败，另附具体细胞/基因差值和第一个失败批次的少量预测数组。
- `matching_coverage.json` / `receiver_matching.csv`：匹配后覆盖率、每个 receiver 的屏蔽数量和两类 sender 移除比例。
- `control_repeats.csv` / `control_summary.csv`：所有重复及均值±样本 SD；MSE 按同一 cohort 的全部 cell–gene 项汇总。
- `paired_contrast_repeats.csv` / `paired_contrast_summary.csv`：tumor_2−tumor_1，以及各 tumor−random 的配对重复差值。没有把细胞当作独立生物重复计算显著性。
- `selected_sender_balance.csv`：实际被选 sender 的距离、模型输入表达强度、可见 residual 强度。
- `new_control_repeat_distributions.png`：新控制全部 10 次重复点及 mean±SD。
- `existing_original_matched_count_repeats.csv` / `existing_matched_count_repeat_distributions.png`：原 sender-origin 分析的十次重复及两项指标分布，来源已内嵌。它使用原始 receiver 集合，不能与新匹配 cohort 直接混作同一次对照。
- `source/`：实际运行的冻结源码与分析脚本，便于复现。

这些结果检验固定模型在指定干预下的预测敏感性。距离分层是近似匹配，不等于精确距离、表达强度和细胞身份都已被完全分离。后续根据实际覆盖率、匹配平衡和两类操作的结果写回复，不预先替换 Figure 5。
